<a href="https://colab.research.google.com/github/matildadesa/LAI_estimation/blob/main/LAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Single pixel NDVI Sentinel 2

In [ ]:
# Sentinel-2 NDVI extraction (single-pixel sampling)

# Processing workflow:
#  Scene cloud prefiltering, keep scenes with ≤ 80% cloud
#  Pixel-level masking combining:
#       Scene Classification Layer (SCL)
#       QA60 cloud and cirrus flags
#       s2cloudless cloud probability threshold
#       60 m dilation to remove cloud-edge contamination
#  Temporal matching to field observations using adaptive windows:
#       ±15 days where ≥ 2 observations are available
#       otherwise expanded to ±30 or ±50 days
#
# Outputs (per plot):
#   NDVI_S2 = median NDVI from valid observations within selected window
#   window_days_used = temporal window applied (15 / 30 / 50 / none)
#   n_images = number of scenes within window
#   n_valid = number of valid observations after pixel-level masking


# Mount Drive

from google.colab import drive
drive.mount("/content/drive")


# Initialise Earth engine

import pandas as pd
import numpy as np
import ee

ee.Authenticate()
ee.Initialize(project="double-skyline-471010-g2")


# Load ground dataset

BASE = "/content/drive/MyDrive/LAI/"
INFILE = BASE + "LAI_GROUND.csv"

df = pd.read_csv(INFILE, parse_dates=["Date"])

df["Landuse"] = df["Landuse"].astype(str).str.strip()
df["LAI"] = pd.to_numeric(df["LAI"], errors="coerce")
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

df = df.dropna(subset=["Date", "LAI", "lat", "lon", "Landuse"]).copy()
df = df[df["Landuse"].isin(["Forest", "Oil palm"])].copy()

# Check data

print("Rows kept:", len(df))
print(df[["Landuse", "LAI", "lat", "lon", "Date"]].head())


# Convert plot data to EE points

def row_to_feature(r):
    geom = ee.Geometry.Point([float(r["lon"]), float(r["lat"])])
    date_str = pd.to_datetime(r["Date"]).strftime("%Y-%m-%d")
    props = {
        "Plot": str(r["Plot"]) if "Plot" in df.columns else None,
        "Landuse": str(r["Landuse"]),
        "LAI": float(r["LAI"]),
        "date_center": date_str
    }
    return ee.Feature(geom, props)

fc = ee.FeatureCollection([row_to_feature(r) for _, r in df.iterrows()])
print("EE features:", fc.size().getInfo())


# Masking setup

S2_SR   = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
S2_CPROB = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")

# parameters to change strictness of masking
MAX_SCENE_CLOUD_PCT = 80
CLOUD_PROB_THR      = 40
DILATE_METERS       = 60
MIN_VALID           = 2

# Join surface with cloud probability
join = ee.Join.saveFirst("cloud_prob_img")
join_filter = ee.Filter.equals(leftField="system:index", rightField="system:index")

s2_joined = join.apply(
    primary=S2_SR,
    secondary=S2_CPROB,
    condition=join_filter
)

def strict_mask_s2(img):

    scl = img.select("SCL")


     # Remove 0 no data, 1 saturated/defective, 2 dark area pixels, 3 cloud shadow,
    # 8 cloud medium prob, 9 cloud high prob, 10 cirrus, 11 snow/ice (kept 11 just in case of misclassification)
    good = (scl.neq(0)
            .And(scl.neq(1))
            .And(scl.neq(2))
            .And(scl.neq(3))
            .And(scl.neq(8))
            .And(scl.neq(9))
            .And(scl.neq(10))
            .And(scl.neq(11)))

    # QA60 mask
    qa = img.select("QA60")
    cloud_bit  = 1 << 10
    cirrus_bit = 1 << 11
    qa_clear = qa.bitwiseAnd(cloud_bit).eq(0).And(qa.bitwiseAnd(cirrus_bit).eq(0))

    # Cloud probability mask
    cprob = ee.Image(img.get("cloud_prob_img")).select("probability")
    cprob_clear = cprob.lt(CLOUD_PROB_THR)

    mask = good.And(qa_clear).And(cprob_clear)

    # Dilation to remove cloud edges/haze fringes
    if DILATE_METERS and DILATE_METERS > 0:
        bad = mask.Not()
        bad_grown = bad.focal_max(radius=DILATE_METERS, units="meters")
        mask = bad_grown.Not()

    return img.updateMask(mask)

def add_ndvi(img):
    red = img.select("B4").multiply(0.0001)
    nir = img.select("B8").multiply(0.0001)
    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI_S2")
    return img.addBands(ndvi).copyProperties(img, ["system:time_start", "CLOUDY_PIXEL_PERCENTAGE"])

# Build NDVI
#  scene prefilter
#  apply pixel mask
#  compute NDVI
s2_ndvi_strict = (ee.ImageCollection(s2_joined)
                  .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", MAX_SCENE_CLOUD_PCT))
                  .map(strict_mask_s2)
                  .map(add_ndvi)
                  .select(["NDVI_S2"]))


# expanding time window to ensure enough valid NDVI observations

def per_image_point_ndvi_fc(ic, geom):
    """Return a FeatureCollection of per-image NDVI at point (non-null only)."""
    def sample_one(img):
        s = img.select("NDVI_S2").sample(region=geom, scale=10, numPixels=1, geometries=False).first()
        v = ee.Algorithms.If(s, ee.Feature(s).get("NDVI_S2"), None)
        return ee.Feature(None, {"v": v})
    return ic.map(sample_one).filter(ee.Filter.notNull(["v"]))

def nvalid_and_median(ic, geom):
    per_img = per_image_point_ndvi_fc(ic, geom)
    n_valid = per_img.size()
    v_list = ee.List(per_img.aggregate_array("v"))
    v_med = ee.Algorithms.If(n_valid.gt(0), ee.Number(v_list.reduce(ee.Reducer.median())), None)
    return n_valid, v_med

def add_s2_ndvi_to_feature(feat):
    center = ee.Date(feat.get("date_center"))
    geom = feat.geometry()

    def ic_for(days):
        return (s2_ndvi_strict
                .filterDate(center.advance(-days, "day"), center.advance(days, "day"))
                .filterBounds(geom))

    ic15, ic30, ic50 = ic_for(15), ic_for(30), ic_for(50)

    # number of scenes in each window
    n15, n30, n50 = ic15.size(), ic30.size(), ic50.size()

    # number of valid observations and median NDVI
    nvalid15, v15 = nvalid_and_median(ic15, geom)
    nvalid30, v30 = nvalid_and_median(ic30, geom)
    nvalid50, v50 = nvalid_and_median(ic50, geom)

    def use(days, n_images, n_valid, ndvi_val):
        return feat.set({
            "window_days_used": days,
            "n_images": n_images,
            "n_valid": n_valid,
            "NDVI_S2": ndvi_val
        })

    # use smallest time window with enough observations, else expand
    return ee.Algorithms.If(
        ee.Number(nvalid15).gte(MIN_VALID),
        use(15, n15, nvalid15, v15),
        ee.Algorithms.If(
            ee.Number(nvalid30).gte(MIN_VALID),
            use(30, n30, nvalid30, v30),
            ee.Algorithms.If(
                ee.Number(nvalid50).gte(MIN_VALID),
                use(50, n50, nvalid50, v50),
                feat.set({"window_days_used": None, "n_images": 0, "n_valid": 0, "NDVI_S2": None})
            )
        )
    )

out_fc_s2 = fc.map(lambda f: ee.Feature(add_s2_ndvi_to_feature(f)))



# Export

OUT_CSV = "S2_NDVI_singlepixel_STRICTmask.csv"

task = ee.batch.Export.table.toDrive(
    collection=out_fc_s2,
    description="export_s2_ndvi_singlepixel_STRICTmask",
    folder="LAI",
    fileNamePrefix=OUT_CSV.replace(".csv", ""),
    fileFormat="CSV"
)
task.start()



# Single pixel NDVI Landsat 8/9

In [ ]:
# Landsat 8/9 NDVI extraction (single pixel sampling)

# Processing workflow:
#  Scene cloud prefiltering keep scenes with ≤ 80% cloud
#  Pixel-level masking using:
#       QA_PIXEL cloud, cirrus, shadow, snow, and fill flags
#       60 m dilation to remove cloud-edge contamination
#  Temporal matching to field observations using adaptive windows:
#       ±15 days where ≥ 2 observations are available
#       otherwise expanded to ±30 or ±50 days
#
# Outputs (per plot):
#   NDVI_L89 = median NDVI from valid observations within selected window
#   window_days_used = temporal window applied (15 / 30 / 50 / none)
#   n_images = number of scenes within window
#   n_valid = number of valid observations after pixel-level masking


# mount Drive

from google.colab import drive
drive.mount("/content/drive")


# initialise Earth Engine

import pandas as pd
import ee

ee.Authenticate()
ee.Initialize(project="double-skyline-471010-g2")


# Load ground dataset

BASE = "/content/drive/MyDrive/LAI/"
INFILE = BASE + "LAI_with_geolocation_WGS84_PADDED.csv"

df = pd.read_csv(INFILE, parse_dates=["Date"])

df["Landuse"] = df["Landuse"].astype(str).str.strip()
df["LAI"] = pd.to_numeric(df["LAI"], errors="coerce")
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

df = df.dropna(subset=["Date", "LAI", "lat", "lon", "Landuse"]).copy()
df = df[df["Landuse"].isin(["Forest", "Oil palm"])].copy()


# Check data

print("Rows kept:", len(df))


# Convert plot data to EE points

def row_to_feature(r):
    geom = ee.Geometry.Point([float(r["lon"]), float(r["lat"])])
    date_str = pd.to_datetime(r["Date"]).strftime("%Y-%m-%d")
    props = {
        "Plot": str(r["Plot"]) if "Plot" in df.columns else None,
        "Landuse": str(r["Landuse"]),
        "LAI": float(r["LAI"]),
        "date_center": date_str
    }
    return ee.Feature(geom, props)

fc = ee.FeatureCollection([row_to_feature(r) for _, r in df.iterrows()])



# Masking setup

MAX_SCENE_CLOUD_PCT = 80
DILATE_METERS       = 60
MIN_VALID           = 2


# load Landsat 8/9 surface reflectance collections

L8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
L9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
L89 = L8.merge(L9)


# scene cloud prefiltering

L89_prefilter = L89.filter(ee.Filter.lte("CLOUD_COVER", MAX_SCENE_CLOUD_PCT))

def strict_mask_l89(img):

    qa = img.select("QA_PIXEL")

    # remove 0 fill, 1 dilated cloud, 2 cirrus, 3 cloud, 4 cloud shadow, 5 snow
    fill          = qa.bitwiseAnd(1 << 0).neq(0)
    dilated_cloud = qa.bitwiseAnd(1 << 1).neq(0)
    cirrus        = qa.bitwiseAnd(1 << 2).neq(0)
    cloud         = qa.bitwiseAnd(1 << 3).neq(0)
    shadow        = qa.bitwiseAnd(1 << 4).neq(0)
    snow          = qa.bitwiseAnd(1 << 5).neq(0)

    # keep only clear pixels
    clear = (fill.Not()
             .And(dilated_cloud.Not())
             .And(cirrus.Not())
             .And(cloud.Not())
             .And(shadow.Not())
             .And(snow.Not()))

    # Dilation to remove cloud edges/haze fringes
    if DILATE_METERS and DILATE_METERS > 0:
        bad = clear.Not()
        bad_grown = bad.focal_max(radius=DILATE_METERS, units="meters")
        clear = bad_grown.Not()

    return img.updateMask(clear)

def add_ndvi_l89(img):
    # C2 L2 surface reflectance bands: SR_B4 = red, SR_B5 = nir
    red = img.select("SR_B4").multiply(0.0000275).add(-0.2)
    nir = img.select("SR_B5").multiply(0.0000275).add(-0.2)
    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI_L89")

    return (img.addBands(ndvi)
              .copyProperties(img, ["system:time_start", "CLOUD_COVER"]))


# Build NDVI
#  scene prefilter
#  apply pixel mask
#  compute NDVI

l89_ndvi_strict = (L89_prefilter
                   .map(strict_mask_l89)
                   .map(add_ndvi_l89)
                   .select(["NDVI_L89"]))


# Expand time window if too few valid observations

def per_image_point_ndvi_fc(ic, geom):
    def sample_one(img):
        s = img.select("NDVI_L89").sample(region=geom, scale=30, numPixels=1, geometries=False).first()
        v = ee.Algorithms.If(s, ee.Feature(s).get("NDVI_L89"), None)
        return ee.Feature(None, {"v": v})


def nvalid_and_median(ic, geom):
    per_img = per_image_point_ndvi_fc(ic, geom)
    n_valid = per_img.size()
    v_list = ee.List(per_img.aggregate_array("v"))
    v_med = ee.Algorithms.If(n_valid.gt(0), ee.Number(v_list.reduce(ee.Reducer.median())), None)
    return n_valid, v_med

def add_l89_ndvi_to_feature(feat):
    center = ee.Date(feat.get("date_center"))
    geom = feat.geometry()

    def ic_for(days):
        return (l89_ndvi_strict
                .filterDate(center.advance(-days, "day"), center.advance(days, "day"))
                .filterBounds(geom))

    ic15, ic30, ic50 = ic_for(15), ic_for(30), ic_for(50)

    # number of scenes in each window
    n15, n30, n50 = ic15.size(), ic30.size(), ic50.size()

    # number of valid observations and median NDVI
    nvalid15, v15 = nvalid_and_median(ic15, geom)
    nvalid30, v30 = nvalid_and_median(ic30, geom)
    nvalid50, v50 = nvalid_and_median(ic50, geom)

    def use(days, n_images, n_valid, ndvi_val):
        return feat.set({
            "window_days_used": days,
            "n_images": n_images,
            "n_valid": n_valid,
            "NDVI_L89": ndvi_val
        })

    # use smallest time window with enough observations, else expand
    return ee.Algorithms.If(
        ee.Number(nvalid15).gte(MIN_VALID),
        use(15, n15, nvalid15, v15),
        ee.Algorithms.If(
            ee.Number(nvalid30).gte(MIN_VALID),
            use(30, n30, nvalid30, v30),
            ee.Algorithms.If(
                ee.Number(nvalid50).gte(MIN_VALID),
                use(50, n50, nvalid50, v50),
                feat.set({"window_days_used": None, "n_images": 0, "n_valid": 0, "NDVI_L89": None})
            )
        )
    )

out_fc_l89 = fc.map(lambda f: ee.Feature(add_l89_ndvi_to_feature(f)))


# Export

OUT_CSV = "L89_NDVI_singlepixel_STRICTmask.csv"

task = ee.batch.Export.table.toDrive(
    collection=out_fc_l89,
    description="export_l89_ndvi_singlepixel_STRICTmask",
    folder="LAI",
    fileNamePrefix=OUT_CSV.replace(".csv", ""),
    fileFormat="CSV"
)
task.start()



# Single pixel red-edge indices Sentinel-2

In [ ]:
# Sentinel-2 red-edge index extraction (single-pixel sampling)

# Processing workflow:
#  Load plots retained after Sentinel-2 NDVI masking
#  rebuild plot locations and use the same selected time window for each plot
#  scene cloud prefiltering, keep scenes with ≤ 80% cloud
#  pixel-level masking combining:
#       Scene Classification Layer (SCL)
#       QA60 cloud and cirrus flags
#       s2cloudless cloud probability threshold
#       60 m dilation to remove cloud-edge contamination
#  Calculates NDVI (again), NDRE705, CIre705, and MTCI
#  Extract median index values at the plot centre using the selected time window
#
# Outputs (per plot):
#   NDVI_new = median NDVI within selected window
#   NDRE705 = median NDRE705 within selected window
#   CIre705 = median CIre705 within selected window
#   MTCI = median MTCI within selected window


import pandas as pd
import ee


# Load sentinel-2 NDVI output

strict_path = "/content/drive/MyDrive/LAI/S2_NDVI_singlepixel_STRICTmask.csv"
df = pd.read_csv(strict_path)


# Clean data and keep plots with valid NDVI and selected window

df["NDVI_S2"] = pd.to_numeric(df["NDVI_S2"], errors="coerce")
df["LAI"] = pd.to_numeric(df["LAI"], errors="coerce")
df["window_days_used"] = pd.to_numeric(df["window_days_used"], errors="coerce")
df["date_center"] = pd.to_datetime(df["date_center"], errors="coerce")

df = df.dropna(subset=["LAI", "NDVI_S2", "window_days_used", ".geo", "date_center"])
df = df[df["Landuse"].isin(["Forest", "Oil palm"])].copy()

print("Rows kept for indices:", len(df))
print(df[["Plot","Landuse","date_center","window_days_used","NDVI_S2"]].head())


import json


# Convert geometry to coordinates

def geo_to_lonlat(geo_str):
    g = json.loads(geo_str)
    lon, lat = g["coordinates"]
    return float(lon), float(lat)


# Convert plot data to EE points

def row_to_feature(r):
    lon, lat = geo_to_lonlat(r[".geo"])
    geom = ee.Geometry.Point([lon, lat])
    return ee.Feature(geom, {
        "Plot": str(r["Plot"]),
        "Landuse": str(r["Landuse"]),
        "LAI": float(r["LAI"]),
        "date_center": r["date_center"].strftime("%Y-%m-%d"),
        "window_days_used": float(r["window_days_used"])
    })

fc = ee.FeatureCollection([row_to_feature(r) for _, r in df.iterrows()])
print("EE features:", fc.size().getInfo())


# Masking setup

S2_SR    = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
S2_CPROB = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")

MAX_SCENE_CLOUD_PCT = 80
CLOUD_PROB_THR      = 40
DILATE_METERS       = 60


# Join surface reflectance with cloud probability

join = ee.Join.saveFirst("cloud_prob_img")
join_filter = ee.Filter.equals(leftField="system:index", rightField="system:index")

s2_joined = join.apply(S2_SR, S2_CPROB, join_filter)

def strict_mask_s2(img):
    scl = img.select("SCL")
    good = (scl.neq(0)
            .And(scl.neq(1))
            .And(scl.neq(2))
            .And(scl.neq(3))
            .And(scl.neq(8))
            .And(scl.neq(9))
            .And(scl.neq(10))
            .And(scl.neq(11)))

    # QA60 mask
    qa = img.select("QA60")
    cloud_bit  = 1 << 10
    cirrus_bit = 1 << 11
    qa_clear = qa.bitwiseAnd(cloud_bit).eq(0).And(qa.bitwiseAnd(cirrus_bit).eq(0))

    # Cloud probability
    cprob = ee.Image(img.get("cloud_prob_img")).select("probability")
    cprob_clear = cprob.lt(CLOUD_PROB_THR)

    mask = good.And(qa_clear).And(cprob_clear)

    # Dilation
    if DILATE_METERS and DILATE_METERS > 0:
        bad = mask.Not()
        bad_grown = bad.focal_max(radius=DILATE_METERS, units="meters")
        mask = bad_grown.Not()

    return img.updateMask(mask)

def add_indices(img):
    red  = img.select("B4").multiply(0.0001)
    nir  = img.select("B8").multiply(0.0001)
    re1  = img.select("B5").multiply(0.0001)
    re2  = img.select("B6").multiply(0.0001)
    re3  = img.select("B7").multiply(0.0001)

    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI")
    ndre = nir.subtract(re1).divide(nir.add(re1)).rename("NDRE705")
    cire = nir.divide(re1).subtract(1).rename("CIre705")
    mtci = re3.subtract(re1).divide(re2.subtract(re1)).rename("MTCI")

    return img.addBands([ndvi, ndre, cire, mtci]).copyProperties(img, ["system:time_start", "CLOUDY_PIXEL_PERCENTAGE"])


# Build red-edge index collection
#  scene prefilter
#  apply pixel mask
#  compute indices

s2_strict_idx = (ee.ImageCollection(s2_joined)
    .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", MAX_SCENE_CLOUD_PCT))
    .map(strict_mask_s2)
    .map(add_indices)
    .select(["NDVI", "NDRE705", "CIre705", "MTCI"])
)


# Extract median index values using selected time window

def add_idx_vals(feat):
    center = ee.Date(feat.get("date_center"))
    geom   = feat.geometry()
    w      = ee.Number(feat.get("window_days_used"))

    ic = (s2_strict_idx
          .filterBounds(geom)
          .filterDate(center.advance(w.multiply(-1), "day"), center.advance(w, "day")))

    img = ic.median()
    s = img.sample(region=geom, scale=10, numPixels=1, geometries=False).first()

    return feat.set({
        "NDVI_new":  ee.Algorithms.If(s, ee.Feature(s).get("NDVI"), None),
        "NDRE705":   ee.Algorithms.If(s, ee.Feature(s).get("NDRE705"), None),
        "CIre705":   ee.Algorithms.If(s, ee.Feature(s).get("CIre705"), None),
        "MTCI":      ee.Algorithms.If(s, ee.Feature(s).get("MTCI"), None),
    })

out_fc = fc.map(add_idx_vals)


import pandas as pd
from scipy.stats import linregress


# Convert output to dataframe

data = pd.DataFrame([f["properties"] for f in out_fc.getInfo()["features"]])

for col in ["LAI", "NDVI_new", "NDRE705", "CIre705", "MTCI"]:
    data[col] = pd.to_numeric(data[col], errors="coerce")

data["Landuse"] = data["Landuse"].astype(str).str.strip()

data2 = data.dropna(subset=["LAI", "NDVI_new", "NDRE705", "CIre705", "MTCI"]).copy()
print("Rows with all indices:", len(data2))


# Report R squared values

def r2_report(df_in, label):
    print(f"\n--- {label} ---")
    for idx in ["NDVI_new", "NDRE705", "CIre705", "MTCI"]:
        r = linregress(df_in[idx].values, df_in["LAI"].values).rvalue
        print(f"{idx}: R²={r**2:.3f} | n={len(df_in)}")

r2_report(data2, "ALL")

for lc in ["Forest", "Oil palm"]:
    r2_report(data2[data2["Landuse"] == lc], lc)

# Area based NDVI Sentinel-2

In [ ]:
# Sentinel-2 NDVI extraction (area-weighted plot mean)

# Processing workflow is the same as single pixel but
#  NDVI is extracted as an area-weighted mean across the 1000 m sqaured plot footprint

# Outputs (per plot):
#   NDVI_S2 = median area-weighted NDVI within selected window
#   window_days_used = temporal window applied (15 / 30 / 50 / none)
#   n_images = number of scenes within window
#   n_valid = number of valid observations after pixel-level masking
#   valid_area_m2_median_date = median valid area within plot footprint
#   pixel_equiv_median_date = median number of contributing 10 m pixels


# Mount Drive

from google.colab import drive
drive.mount("/content/drive")


# Initialise Earth Engine

import pandas as pd
import numpy as np
import ee
import time

try:
    ee.Initialize(project="double-skyline-471010-g2")
except:
    ee.Authenticate()
    ee.Initialize(project="double-skyline-471010-g2")


# Load ground dataset

BASE = "/content/drive/MyDrive/LAI/"
INFILE = BASE + "LAI_GROUND.csv"

df = pd.read_csv(INFILE)

df["Plot"] = df["Plot"].astype(str).str.strip()
df["Landuse"] = df["Landuse"].astype(str).str.strip()
df["LAI"] = pd.to_numeric(df["LAI"], errors="coerce")
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["date_center"] = pd.to_datetime(df["date_center"], errors="coerce")

df = df.dropna(subset=["Plot", "Landuse", "LAI", "lat", "lon", "Date", "date_center"]).copy()
df = df[df["Landuse"].isin(["Forest", "Oil palm"])].copy()

# Convert plot data to EE points

def row_to_feature(r):
    geom = ee.Geometry.Point([float(r["lon"]), float(r["lat"])])
    date_str = pd.to_datetime(r["date_center"]).strftime("%Y-%m-%d")

    props = {
        "Plot": str(r["Plot"]),
        "Landuse": str(r["Landuse"]),
        "LAI": float(r["LAI"]),
        "date_center": date_str
    }
    return ee.Feature(geom, props)

fc = ee.FeatureCollection([row_to_feature(r) for _, r in df.iterrows()])



# Masking setup

S2_SR = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
S2_CPROB = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")

MAX_SCENE_CLOUD_PCT = 80
CLOUD_PROB_THR = 40
DILATE_METERS = 60
MIN_VALID = 2


# Plot footprint setup

PLOT_AREA_M2 = 1000.0
PLOT_RADIUS_M = (PLOT_AREA_M2 / np.pi) ** 0.5
SCALE = 10
FINE_SCALE = 1

print(f"Plot buffer radius: {PLOT_RADIUS_M:.2f} m")


# Join surface reflectance with cloud probability

join = ee.Join.saveFirst("cloud_prob_img")
join_filter = ee.Filter.equals(leftField="system:index", rightField="system:index")

s2_joined = join.apply(
    primary=S2_SR,
    secondary=S2_CPROB,
    condition=join_filter
)

def strict_mask_s2(img):
    scl = img.select("SCL")

    good = (
        scl.neq(0)
        .And(scl.neq(1))
        .And(scl.neq(2))
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    # QA60 mask
    qa = img.select("QA60")
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11

    qa_clear = qa.bitwiseAnd(cloud_bit).eq(0).And(
        qa.bitwiseAnd(cirrus_bit).eq(0)
    )

    # Cloud probability mask
    cprob = ee.Image(img.get("cloud_prob_img")).select("probability")
    cprob_clear = cprob.lt(CLOUD_PROB_THR)

    mask = good.And(qa_clear).And(cprob_clear)

    # Dilation
    if DILATE_METERS > 0:
        bad = mask.Not()
        bad = bad.focal_max(radius=DILATE_METERS, units="meters")
        mask = bad.Not()

    return img.updateMask(mask)

def add_ndvi(img):
    red = img.select("B4").multiply(0.0001)
    nir = img.select("B8").multiply(0.0001)
    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI_S2")

    return img.addBands(ndvi).copyProperties(
        img, ["system:time_start", "CLOUDY_PIXEL_PERCENTAGE"]
    )


# build NDVI

s2_ndvi_strict = (
    ee.ImageCollection(s2_joined)
    .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", MAX_SCENE_CLOUD_PCT))
    .map(strict_mask_s2)
    .map(add_ndvi)
    .select(["NDVI_S2"])
)


# Area weighted extraction across plot footprint

def per_image_plot_ndvi_fc(ic, geom):
    plot_geom = geom.buffer(PLOT_RADIUS_M)

    def sample_one(img):
        ndvi = img.select("NDVI_S2")
        proj = ndvi.projection()

        inside_fine = (
            ee.Image.constant(1)
            .toFloat()
            .clip(plot_geom)
            .reproject(crs=proj, scale=FINE_SCALE)
        )

        inside_frac = (
            inside_fine
            .reduceResolution(
                reducer=ee.Reducer.mean(),
                maxPixels=4096
            )
            .reproject(crs=proj, scale=SCALE)
            .rename("inside_frac")
        )

        pixel_area = ee.Image.pixelArea().reproject(crs=proj, scale=SCALE)
        weights = inside_frac.multiply(pixel_area).rename("weights")

        valid_weights = weights.updateMask(ndvi.mask()).rename("weights")
        weighted_ndvi = ndvi.multiply(valid_weights).rename("weighted_ndvi")

        stats = ee.Image.cat([weighted_ndvi, valid_weights]).reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=plot_geom,
            scale=SCALE,
            crs=proj,
            maxPixels=1e7
        )

        num_raw = stats.get("weighted_ndvi")
        den_raw = stats.get("weights")

        num = ee.Number(ee.Algorithms.If(num_raw, num_raw, 0))
        den = ee.Number(ee.Algorithms.If(den_raw, den_raw, 0))

        v = ee.Algorithms.If(den.gt(0), num.divide(den), None)
        valid_area_m2 = ee.Algorithms.If(den.gt(0), den, None)
        pixel_equiv = ee.Algorithms.If(den.gt(0), den.divide(SCALE * SCALE), None)

        return ee.Feature(None, {
            "v": v,
            "valid_area_m2": valid_area_m2,
            "pixel_equiv": pixel_equiv
        })

    return ic.map(sample_one).filter(ee.Filter.notNull(["v"]))

def nvalid_and_median(ic, geom):
    per_img = per_image_plot_ndvi_fc(ic, geom)

    n_valid = per_img.size()

    v_list = ee.List(per_img.aggregate_array("v"))
    area_list = ee.List(per_img.aggregate_array("valid_area_m2"))
    pixeq_list = ee.List(per_img.aggregate_array("pixel_equiv"))

    v_med = ee.Algorithms.If(
        n_valid.gt(0),
        ee.Number(v_list.reduce(ee.Reducer.median())),
        None
    )

    area_med = ee.Algorithms.If(
        n_valid.gt(0),
        ee.Number(area_list.reduce(ee.Reducer.median())),
        None
    )

    pixeq_med = ee.Algorithms.If(
        n_valid.gt(0),
        ee.Number(pixeq_list.reduce(ee.Reducer.median())),
        None
    )

    return n_valid, v_med, area_med, pixeq_med

def add_s2_ndvi_to_feature(feat):
    center = ee.Date(feat.get("date_center"))
    geom = feat.geometry()

    def ic_for(days):
        days = ee.Number(days)
        return (
            s2_ndvi_strict
            .filterDate(center.advance(days.multiply(-1), "day"),
                        center.advance(days, "day"))
            .filterBounds(geom)
        )

    ic15 = ic_for(15)
    ic30 = ic_for(30)
    ic50 = ic_for(50)

    # Number of scenes in each window
    n15 = ic15.size()
    n30 = ic30.size()
    n50 = ic50.size()

    # Number of valid observations, median NDVI, and median valid area
    nvalid15, v15, area15, pixeq15 = nvalid_and_median(ic15, geom)
    nvalid30, v30, area30, pixeq30 = nvalid_and_median(ic30, geom)
    nvalid50, v50, area50, pixeq50 = nvalid_and_median(ic50, geom)

    def use(days, n_images, n_valid, ndvi_val, area_med, pixeq_med):
        return feat.set({
            "window_days_used": days,
            "n_images": n_images,
            "n_valid": n_valid,
            "valid_area_m2_median_date": area_med,
            "pixel_equiv_median_date": pixeq_med,
            "NDVI_S2": ndvi_val
        })

    # Use smallest time window with enough observations
    return ee.Algorithms.If(
        ee.Number(nvalid15).gte(MIN_VALID),
        use(15, n15, nvalid15, v15, area15, pixeq15),
        ee.Algorithms.If(
            ee.Number(nvalid30).gte(MIN_VALID),
            use(30, n30, nvalid30, v30, area30, pixeq30),
            ee.Algorithms.If(
                ee.Number(nvalid50).gte(MIN_VALID),
                use(50, n50, nvalid50, v50, area50, pixeq50),
                feat.set({
                    "window_days_used": None,
                    "n_images": 0,
                    "n_valid": 0,
                    "valid_area_m2_median_date": None,
                    "pixel_equiv_median_date": None,
                    "NDVI_S2": None
                })
            )
        )
    )

out_fc_s2 = fc.map(lambda f: ee.Feature(add_s2_ndvi_to_feature(f)))

# Export

OUT_CSV = "S2_NDVI_areawtd_STRICTmask"

task = ee.batch.Export.table.toDrive(
    collection=out_fc_s2,
    description="export_s2_ndvi_areawtd_STRICTmask",
    folder="LAI",
    fileNamePrefix=OUT_CSV,
    fileFormat="CSV"
)

task.start()



# Area based NDVI Landsat 8/9

In [ ]:
# Landsat 8/9 NDVI extraction (area-weighted plot mean)

# Processing workflow same as single pixel except
#  NDVI extracted as an area-weighted mean across the 1000 m plot footprint
#
# Outputs (per plot):
#   NDVI_L89 = median area-weighted NDVI within selected window
#   window_days_used = temporal window applied (15 / 30 / 50 / none)
#   n_images = number of scenes within window
#   n_valid = number of valid observations after pixel-level masking
#   valid_area_m2 = median valid area within plot footprint


# Mount Drive

from google.colab import drive
drive.mount("/content/drive")


# Earth Engine

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress
import ee

try:
    ee.Initialize(project="double-skyline-471010-g2")
except:
    ee.Authenticate()
    ee.Initialize(project="double-skyline-471010-g2")


# Set paths

BASE_FOLDER = "/content/drive/MyDrive/LAI"
GROUND_PATH = os.path.join(BASE_FOLDER, "LAI_GROUND.csv")
OUT_PATH = os.path.join(BASE_FOLDER, "L89_NDVI_weighted_1000m2_STRICTmask.csv")


# Load ground dataset

df = pd.read_csv(GROUND_PATH)

df["Plot"] = df["Plot"].astype(str).str.strip()
df["Landuse"] = df["Landuse"].astype(str).str.strip()
df["LAI"] = pd.to_numeric(df["LAI"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")

if "date_center" in df.columns:
    df["date_center"] = pd.to_datetime(df["date_center"], errors="coerce")
else:
    df["date_center"] = pd.to_datetime(df["Date"], errors="coerce")

df = df[df["Landuse"].isin(["Forest", "Oil palm"])].copy()
df = df.dropna(subset=["Plot", "Landuse", "LAI", "lon", "lat", "date_center"]).copy()


# Plot footprint setup

PLOT_AREA_M2 = 1000.0
PLOT_RADIUS_M = float(np.sqrt(PLOT_AREA_M2 / np.pi))




# Convert plot data to EE plot footprints

def row_to_feature(r):
    point = ee.Geometry.Point([float(r["lon"]), float(r["lat"])])
    plot_circle = point.buffer(PLOT_RADIUS_M)

    return ee.Feature(plot_circle, {
        "Plot": str(r["Plot"]),
        "Landuse": str(r["Landuse"]),
        "LAI": float(r["LAI"]),
        "date_center": pd.to_datetime(r["date_center"]).strftime("%Y-%m-%d"),
        "lon": float(r["lon"]),
        "lat": float(r["lat"]),
        "plot_area_m2_target": PLOT_AREA_M2,
        "plot_radius_m": PLOT_RADIUS_M
    })

fc = ee.FeatureCollection([row_to_feature(r) for _, r in df.iterrows()])



# Masking setup

MAX_SCENE_CLOUD_PCT = 80
DILATE_METERS = 60
MIN_VALID = 2

L8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
L9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
L89 = L8.merge(L9)


# Scene cloud initial filtering

L89_prefilter = L89.filter(ee.Filter.lte("CLOUD_COVER", MAX_SCENE_CLOUD_PCT))

def strict_mask_l89(img):
    qa = img.select("QA_PIXEL")

    fill = qa.bitwiseAnd(1 << 0).neq(0)
    dilated_cloud = qa.bitwiseAnd(1 << 1).neq(0)
    cirrus = qa.bitwiseAnd(1 << 2).neq(0)
    cloud = qa.bitwiseAnd(1 << 3).neq(0)
    shadow = qa.bitwiseAnd(1 << 4).neq(0)
    snow = qa.bitwiseAnd(1 << 5).neq(0)

    # Keep only clear pixels
    clear = (
        fill.Not()
        .And(dilated_cloud.Not())
        .And(cirrus.Not())
        .And(cloud.Not())
        .And(shadow.Not())
        .And(snow.Not())
    )

    # cloud dilation
    if DILATE_METERS and DILATE_METERS > 0:
        bad = clear.Not()
        bad_grown = bad.focal_max(radius=DILATE_METERS, units="meters")
        clear = bad_grown.Not()

    return img.updateMask(clear)

def add_ndvi_l89(img):
    # Landsat Collection 2 Level 2 scale factors
    red = img.select("SR_B4").multiply(0.0000275).add(-0.2)
    nir = img.select("SR_B5").multiply(0.0000275).add(-0.2)

    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI_L89")

    return img.addBands(ndvi).copyProperties(
        img, ["system:time_start", "CLOUD_COVER"]
    )


# build NDVI
# scene prefilter
# apply pixel mask
# compute NDVI

l89_ndvi_strict = (
    L89_prefilter
    .map(strict_mask_l89)
    .map(add_ndvi_l89)
    .select(["NDVI_L89"])
)


# area-weighted extraction across plot footprint

def weighted_mean_in_geometry(img, geom, band_name, scale):
    band = img.select(band_name)

    area = ee.Image.pixelArea().updateMask(band.mask()).rename("w")
    numerator = band.multiply(area).rename("num")

    stats = ee.Image.cat([numerator, area]).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geom,
        scale=scale,
        maxPixels=1e9,
        bestEffort=True
    )

    num_raw = stats.get("num")
    den_raw = stats.get("w")

    num = ee.Number(ee.Algorithms.If(num_raw, num_raw, 0))
    den = ee.Number(ee.Algorithms.If(den_raw, den_raw, 0))

    value = ee.Algorithms.If(den.gt(0), num.divide(den), None)
    valid_area = ee.Algorithms.If(den.gt(0), den, None)

    return value, valid_area

def per_image_plot_ndvi_fc(ic, geom):
    def sample_one(img):
        v, a = weighted_mean_in_geometry(img, geom, "NDVI_L89", 30)

        return ee.Feature(None, {
            "v": v,
            "valid_area_m2": a
        })


def nvalid_and_median(ic, geom):
    per_img = per_image_plot_ndvi_fc(ic, geom)

    n_valid = per_img.size()
    v_list = ee.List(per_img.aggregate_array("v"))
    a_list = ee.List(per_img.aggregate_array("valid_area_m2"))

    v_med = ee.Algorithms.If(
        n_valid.gt(0),
        ee.Number(v_list.reduce(ee.Reducer.median())),
        None
    )

    a_med = ee.Algorithms.If(
        n_valid.gt(0),
        ee.Number(a_list.reduce(ee.Reducer.median())),
        None
    )

    return n_valid, v_med, a_med


# Expand time window if too few valid observations

def add_l89_ndvi_to_feature(feat):
    center = ee.Date(feat.get("date_center"))
    geom = feat.geometry()

    def ic_for(days):
        days = ee.Number(days)
        return (
            l89_ndvi_strict
            .filterDate(center.advance(days.multiply(-1), "day"),
                        center.advance(days, "day"))
            .filterBounds(geom)
        )

    ic15 = ic_for(15)
    ic30 = ic_for(30)
    ic50 = ic_for(50)

    # Number of scenes in each window
    n15 = ic15.size()
    n30 = ic30.size()
    n50 = ic50.size()

    # Number of valid observations, median NDVI, and median valid area
    nvalid15, v15, area15 = nvalid_and_median(ic15, geom)
    nvalid30, v30, area30 = nvalid_and_median(ic30, geom)
    nvalid50, v50, area50 = nvalid_and_median(ic50, geom)

    def use(days, n_images, n_valid, ndvi_val, area_val):
        return feat.set({
            "window_days_used": days,
            "n_images": n_images,
            "n_valid": n_valid,
            "valid_area_m2": area_val,
            "NDVI_L89": ndvi_val
        })

    # Use smallest time window with enough observations
    return ee.Algorithms.If(
        ee.Number(nvalid15).gte(MIN_VALID),
        use(15, n15, nvalid15, v15, area15),
        ee.Algorithms.If(
            ee.Number(nvalid30).gte(MIN_VALID),
            use(30, n30, nvalid30, v30, area30),
            ee.Algorithms.If(
                ee.Number(nvalid50).gte(MIN_VALID),
                use(50, n50, nvalid50, v50, area50),
                feat.set({
                    "window_days_used": None,
                    "n_images": 0,
                    "n_valid": 0,
                    "valid_area_m2": None,
                    "NDVI_L89": None
                })
            )
        )
    )

out_fc_l89 = fc.map(lambda f: ee.Feature(add_l89_ndvi_to_feature(f)))



# Convert output to dataframe

data = pd.DataFrame([f["properties"] for f in out_fc_l89.getInfo()["features"]])

for col in ["LAI", "lon", "lat", "window_days_used", "n_images", "n_valid", "valid_area_m2", "NDVI_L89"]:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

data["Landuse"] = data["Landuse"].astype(str).str.strip()


# Save output

data.to_csv(OUT_PATH, index=False)


# Area based red-edge Sentinel-2

In [ ]:
# Sentinel-2 red-edge index extraction (area weighted plot mean)

# Processing workflow sam as single pixel except its area weighted now

# Outputs (per plot)
#   NDVI_weighted = median weighted NDVI within selected window
#   NDRE705_weighted = median weighted NDRE705 within selected window
#   CIre705_weighted = median weighted CIre705 within selected window
#   MTCI_weighted = median weighted MTCI within selected window
#   window_days_used = temporal window applied (15 / 30 / 50 / none)
#   n_images_in_window = number of scenes within window
#   n_valid_ndvi = number of valid NDVI observations after pixel level masking
#   valid_area_m2 = median valid area within plot footprint


# Mount Drive

from google.colab import drive
drive.mount("/content/drive")


# Initialise Earth Engine

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress
import ee

try:
    ee.Initialize(project="double-skyline-471010-g2")
except:
    ee.Authenticate()
    ee.Initialize(project="double-skyline-471010-g2")


# Set paths

BASE_FOLDER = "/content/drive/MyDrive/LAI"
GROUND_PATH = os.path.join(BASE_FOLDER, "LAI_GROUND.csv")
OUT_PATH = os.path.join(BASE_FOLDER, "S2_STRICT_weighted_1000m2.csv")


# Load ground dataset

df = pd.read_csv(GROUND_PATH)

df["Plot"] = df["Plot"].astype(str).str.strip()
df["Landuse"] = df["Landuse"].astype(str).str.strip()
df["LAI"] = pd.to_numeric(df["LAI"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")

# Format date_center
df["date_center"] = pd.to_datetime(df["date_center"], errors="coerce")


# plot footprint setup

PLOT_AREA_M2 = 1000.0
PLOT_RADIUS_M = float(np.sqrt(PLOT_AREA_M2 / np.pi))


# Convert plot data to EE plot footprints

def row_to_feature(r):
    point = ee.Geometry.Point([float(r["lon"]), float(r["lat"])])
    plot_circle = point.buffer(PLOT_RADIUS_M)

    return ee.Feature(plot_circle, {
        "Plot": str(r["Plot"]),
        "Landuse": str(r["Landuse"]),
        "LAI": float(r["LAI"]),
        "date_center": pd.to_datetime(r["date_center"]).strftime("%Y-%m-%d"),
        "lon": float(r["lon"]),
        "lat": float(r["lat"]),
        "plot_area_m2_target": PLOT_AREA_M2,
        "plot_radius_m": PLOT_RADIUS_M
    })

fc = ee.FeatureCollection([row_to_feature(r) for _, r in df.iterrows()])


# Masking setup

S2_SR = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
S2_CPROB = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")

MAX_SCENE_CLOUD_PCT = 80
CLOUD_PROB_THR = 40
DILATE_METERS = 60
MIN_VALID = 2

join = ee.Join.saveFirst("cloud_prob_img")
join_filter = ee.Filter.equals(leftField="system:index", rightField="system:index")
s2_joined = join.apply(S2_SR, S2_CPROB, join_filter)

def strict_mask_s2(img):
    scl = img.select("SCL")
    good = (
        scl.neq(0)
        .And(scl.neq(1))
        .And(scl.neq(2))
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    # QA60 mask
    qa = img.select("QA60")
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11
    qa_clear = qa.bitwiseAnd(cloud_bit).eq(0).And(qa.bitwiseAnd(cirrus_bit).eq(0))

    # Cloud probability mask
    cprob = ee.Image(img.get("cloud_prob_img")).select("probability")
    cprob_clear = cprob.lt(CLOUD_PROB_THR)

    mask = good.And(qa_clear).And(cprob_clear)

    # cloud dilation
    if DILATE_METERS > 0:
        bad = mask.Not()
        bad_grown = bad.focal_max(radius=DILATE_METERS, units="meters")
        mask = bad_grown.Not()

    return img.updateMask(mask)

def add_indices(img):
    red = img.select("B4").multiply(0.0001)
    nir = img.select("B8").multiply(0.0001)
    re1 = img.select("B5").multiply(0.0001)
    re2 = img.select("B6").multiply(0.0001)
    re3 = img.select("B7").multiply(0.0001)

    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI")
    ndre = nir.subtract(re1).divide(nir.add(re1)).rename("NDRE705")
    cire = nir.divide(re1).subtract(1).rename("CIre705")
    mtci = re3.subtract(re1).divide(re2.subtract(re1)).rename("MTCI")

    return img.addBands([ndvi, ndre, cire, mtci]).copyProperties(
        img, ["system:time_start", "CLOUDY_PIXEL_PERCENTAGE"]
    )


# compute indices

s2_strict_idx = (
    ee.ImageCollection(s2_joined)
    .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", MAX_SCENE_CLOUD_PCT))
    .map(strict_mask_s2)
    .map(add_indices)
    .select(["NDVI", "NDRE705", "CIre705", "MTCI"])
)


# Area weighted extraction across plot footprint

def weighted_mean_in_geometry(img, geom, band_name, scale):
    band = img.select(band_name)

    area = ee.Image.pixelArea().updateMask(band.mask()).rename("w")
    numerator = band.multiply(area).rename("num")

    stats = ee.Image.cat([numerator, area]).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geom,
        scale=scale,
        maxPixels=1e9,
        bestEffort=True
    )

    num_raw = stats.get("num")
    den_raw = stats.get("w")

    num = ee.Number(ee.Algorithms.If(num_raw, num_raw, 0))
    den = ee.Number(ee.Algorithms.If(den_raw, den_raw, 0))

    value = ee.Algorithms.If(den.gt(0), num.divide(den), None)
    valid_area = ee.Algorithms.If(den.gt(0), den, None)

    return value, valid_area

def per_image_plot_index_fc(ic, geom, band_name, scale):
    def sample_one(img):
        v, a = weighted_mean_in_geometry(img, geom, band_name, scale)
        return ee.Feature(None, {
            "v": v,
            "valid_area_m2": a
        })

    return ic.map(sample_one).filter(ee.Filter.notNull(["v"]))

def median_over_valid_dates(ic, geom, band_name, scale):
    per_img = per_image_plot_index_fc(ic, geom, band_name, scale)
    n_valid = per_img.size()

    v_list = ee.List(per_img.aggregate_array("v"))
    a_list = ee.List(per_img.aggregate_array("valid_area_m2"))

    v_med = ee.Algorithms.If(
        n_valid.gt(0),
        ee.Number(v_list.reduce(ee.Reducer.median())),
        None
    )

    a_med = ee.Algorithms.If(
        n_valid.gt(0),
        ee.Number(a_list.reduce(ee.Reducer.median())),
        None
    )

    return n_valid, v_med, a_med


# Expand time window if more obs needed

def add_idx_vals(feat):
    center = ee.Date(feat.get("date_center"))
    geom = feat.geometry()

    def window_collection(days):
        days = ee.Number(days)
        return (
            s2_strict_idx
            .filterBounds(geom)
            .filterDate(
                center.advance(days.multiply(-1), "day"),
                center.advance(days, "day")
            )
        )

    ic15 = window_collection(15)
    ic30 = window_collection(30)
    ic50 = window_collection(50)

    # Number of scenes in each window
    n15 = ic15.size()
    n30 = ic30.size()
    n50 = ic50.size()

    # Use NDVI to choose time window
    nvalid15, ndvi15, area15 = median_over_valid_dates(ic15, geom, "NDVI", 10)
    nvalid30, ndvi30, area30 = median_over_valid_dates(ic30, geom, "NDVI", 10)
    nvalid50, ndvi50, area50 = median_over_valid_dates(ic50, geom, "NDVI", 10)

    # Extract other indices from same window logic
    _, ndre15, _ = median_over_valid_dates(ic15, geom, "NDRE705", 20)
    _, cire15, _ = median_over_valid_dates(ic15, geom, "CIre705", 20)
    _, mtci15, _ = median_over_valid_dates(ic15, geom, "MTCI", 20)

    _, ndre30, _ = median_over_valid_dates(ic30, geom, "NDRE705", 20)
    _, cire30, _ = median_over_valid_dates(ic30, geom, "CIre705", 20)
    _, mtci30, _ = median_over_valid_dates(ic30, geom, "MTCI", 20)

    _, ndre50, _ = median_over_valid_dates(ic50, geom, "NDRE705", 20)
    _, cire50, _ = median_over_valid_dates(ic50, geom, "CIre705", 20)
    _, mtci50, _ = median_over_valid_dates(ic50, geom, "MTCI", 20)

    return ee.Algorithms.If(
        ee.Number(nvalid15).gte(MIN_VALID),
        feat.set({
            "window_days_used": 15,
            "n_images_in_window": n15,
            "n_valid_ndvi": nvalid15,
            "NDVI_weighted": ndvi15,
            "NDRE705_weighted": ndre15,
            "CIre705_weighted": cire15,
            "MTCI_weighted": mtci15,
            "valid_area_m2": area15
        }),
        ee.Algorithms.If(
            ee.Number(nvalid30).gte(MIN_VALID),
            feat.set({
                "window_days_used": 30,
                "n_images_in_window": n30,
                "n_valid_ndvi": nvalid30,
                "NDVI_weighted": ndvi30,
                "NDRE705_weighted": ndre30,
                "CIre705_weighted": cire30,
                "MTCI_weighted": mtci30,
                "valid_area_m2": area30
            }),
            ee.Algorithms.If(
                ee.Number(nvalid50).gte(MIN_VALID),
                feat.set({
                    "window_days_used": 50,
                    "n_images_in_window": n50,
                    "n_valid_ndvi": nvalid50,
                    "NDVI_weighted": ndvi50,
                    "NDRE705_weighted": ndre50,
                    "CIre705_weighted": cire50,
                    "MTCI_weighted": mtci50,
                    "valid_area_m2": area50
                }),
                feat.set({
                    "window_days_used": None,
                    "n_images_in_window": 0,
                    "n_valid_ndvi": 0,
                    "NDVI_weighted": None,
                    "NDRE705_weighted": None,
                    "CIre705_weighted": None,
                    "MTCI_weighted": None,
                    "valid_area_m2": None
                })
            )
        )
    )

out_fc = fc.map(lambda f: ee.Feature(add_idx_vals(f)))


# Convert output to dataframe

data = pd.DataFrame([f["properties"] for f in out_fc.getInfo()["features"]])

for col in [
    "LAI",
    "lon",
    "lat",
    "window_days_used",
    "n_images_in_window",
    "n_valid_ndvi",
    "NDVI_weighted",
    "NDRE705_weighted",
    "CIre705_weighted",
    "MTCI_weighted",
    "valid_area_m2"
]:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

data["Landuse"] = data["Landuse"].astype(str).str.strip()


# Save full output

data.to_csv(OUT_PATH, index=False)


# five fold cross validation

In [ ]:
# 5-fold cross-validation by plot

# Processing workflow:
#  Load Sentinel-2 single-pixel and area-weighted datasets
#  Run grouped cross-validation using plot as the grouping variable
#  Fit linear models separately for each land use
#  Calculate cross-validated R sqaured and RMSE for each index and method
#
# Outputs:
#   CV_R2 = cross-validated R²
#   CV_RMSE = cross-validated RMSE
#   n_observations = number of valid observations
#   n_plots = number of unique plots
#   n_folds_used = number of folds used in cross-validation


from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

BASE = "/content/drive/MyDrive/LAI/"


# Load datasets

pixel = pd.read_csv(os.path.join(BASE, "S2_STRICT_singlepixel_rededge.csv"))
area  = pd.read_csv(os.path.join(BASE, "S2_STRICT_weighted_1000m2.csv"))


# Clean data

for df in [pixel, area]:
    df["Plot"] = df["Plot"].astype(str).str.strip()
    df["Landuse"] = df["Landuse"].astype(str).str.strip()
    df["LAI"] = pd.to_numeric(df["LAI"], errors="coerce")

# Pixel columns
pixel_cols = ["NDVI_pixel", "NDRE705_pixel", "CIre705_pixel", "MTCI_pixel"]
for col in pixel_cols:
    if col in pixel.columns:
        pixel[col] = pd.to_numeric(pixel[col], errors="coerce")

# Area columns
area_cols = ["NDVI_weighted", "NDRE705_weighted", "CIre705_weighted", "MTCI_weighted"]
for col in area_cols:
    if col in area.columns:
        area[col] = pd.to_numeric(area[col], errors="coerce")


# Cross-validation function

def grouped_cv(df, x_col, index_name, method_name):
    rows = []

    for lc in ["Forest", "Oil palm"]:
        sub = df[df["Landuse"] == lc].dropna(subset=[x_col, "LAI", "Plot"]).copy()

        n_obs = len(sub)
        n_plots = sub["Plot"].nunique()

        if n_obs < 2 or n_plots < 2:
            rows.append({
                "Index": index_name,
                "Land use": lc,
                "Method": method_name,
                "n_observations": n_obs,
                "n_plots": n_plots,
                "n_folds_used": np.nan,
                "CV_R2": np.nan,
                "CV_RMSE": np.nan
            })
            continue

        # Use 5 folds
        n_splits = min(5, n_plots)

        X = sub[[x_col]].values
        y = sub["LAI"].values
        groups = sub["Plot"].values

        gkf = GroupKFold(n_splits=n_splits)

        y_true_all = []
        y_pred_all = []

        for train_idx, test_idx in gkf.split(X, y, groups):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            model = LinearRegression()
            model.fit(X_train, y_train)

            y_pred = model.predict(X_test)

            y_true_all.extend(y_test)
            y_pred_all.extend(y_pred)

        y_true_all = np.array(y_true_all)
        y_pred_all = np.array(y_pred_all)

        rmse = np.sqrt(mean_squared_error(y_true_all, y_pred_all))
        r2 = r2_score(y_true_all, y_pred_all)

        rows.append({
            "Index": index_name,
            "Land use": lc,
            "Method": method_name,
            "n_observations": n_obs,
            "n_plots": n_plots,
            "n_folds_used": n_splits,
            "CV_R2": r2,
            "CV_RMSE": rmse
        })

    return pd.DataFrame(rows)


# Run all index and method combinations

results = []

# Pixel method
results.append(grouped_cv(pixel, "NDVI_pixel", "NDVI", "Pixel"))
results.append(grouped_cv(pixel, "NDRE705_pixel", "NDRE705", "Pixel"))
results.append(grouped_cv(pixel, "CIre705_pixel", "CIre705", "Pixel"))
results.append(grouped_cv(pixel, "MTCI_pixel", "MTCI", "Pixel"))

# Area method
results.append(grouped_cv(area, "NDVI_weighted", "NDVI", "Area"))
results.append(grouped_cv(area, "NDRE705_weighted", "NDRE705", "Area"))
results.append(grouped_cv(area, "CIre705_weighted", "CIre705", "Area"))
results.append(grouped_cv(area, "MTCI_weighted", "MTCI", "Area"))

cv_summary = pd.concat(results, ignore_index=True)


# Sort and round

index_order = {"NDVI": 0, "NDRE705": 1, "CIre705": 2, "MTCI": 3}
landuse_order = {"Forest": 0, "Oil palm": 1}
method_order = {"Pixel": 0, "Area": 1}

cv_summary["_index_order"] = cv_summary["Index"].map(index_order)
cv_summary["_landuse_order"] = cv_summary["Land use"].map(landuse_order)
cv_summary["_method_order"] = cv_summary["Method"].map(method_order)

cv_summary = cv_summary.sort_values(
    ["_index_order", "_landuse_order", "_method_order"]
).drop(columns=["_index_order", "_landuse_order", "_method_order"])

cv_summary["CV_R2"] = cv_summary["CV_R2"].round(3)
cv_summary["CV_RMSE"] = cv_summary["CV_RMSE"].round(3)


# Print

print("\n5-fold cross-validation results:\n")
print(cv_summary.to_string(index=False))


# Save

out_path = os.path.join(BASE, "fivefold_cv_summary_rededge_all_methods.csv")
cv_summary.to_csv(out_path, index=False)



# t-tests

In [ ]:
# Sensor comparison using absolute prediction error

# Processing workflow:
#  Load ground, Sentinel-2, and Landsat NDVI datasets
#  Fit separate LAI - NDVI models by land use
#  Calculate absolute prediction error for each sensor
#  Compare sensors using:
#       Paired t-test on exact matched area-weighted subset
#       Welch unpaired t-test on full valid samples
#
# Outputs:
#   Mean abs. error S2 = mean absolute error for Sentinel-2
#   Mean abs. error L89 = mean absolute error for Landsat 8/9
#   t-statistic = t-test statistic
#   p-value = test significance


from google.colab import drive
drive.mount("/content/drive")

import os
import numpy as np
import pandas as pd
from scipy.stats import linregress, ttest_rel, ttest_ind

BASE = "/content/drive/MyDrive/LAI/"


# Load datasets

ground = pd.read_csv(os.path.join(BASE, "LAI_GROUND.csv"))

# Sentinel-2
s2_area  = pd.read_csv(os.path.join(BASE, "S2_NDVI_areawtd_STRICTmask_valid_only.csv"))
s2_pixel = pd.read_csv(os.path.join(BASE, "Copy of S2_valid_plots_used.csv"))

# Landsat
l89_area  = pd.read_csv(os.path.join(BASE, "L89_valid_plots_used.csv"))
l89_pixel = pd.read_csv(os.path.join(BASE, "Copy of L89_singlepixel_valid_plots_used.csv"))

# Exact paired subset for area-weighted comparison
paired_area = pd.read_csv(os.path.join(BASE, "S2_L89_exact_paired_subset.csv"))


# Clean data

for df in [ground, s2_area, s2_pixel, l89_area, l89_pixel, paired_area]:
    df["Plot"] = df["Plot"].astype(str).str.strip()
    df["Landuse"] = df["Landuse"].astype(str).str.strip()

ground["LAI"] = pd.to_numeric(ground["LAI"], errors="coerce")

s2_area["NDVI_S2"] = pd.to_numeric(s2_area["NDVI_S2"], errors="coerce")
s2_pixel["NDVI_S2"] = pd.to_numeric(s2_pixel["NDVI_S2"], errors="coerce")

l89_area["NDVI_L89"] = pd.to_numeric(l89_area["NDVI_L89"], errors="coerce")
l89_pixel["NDVI_L89"] = pd.to_numeric(l89_pixel["NDVI_L89"], errors="coerce")

paired_area["LAI"] = pd.to_numeric(paired_area["LAI"], errors="coerce")
paired_area["NDVI_S2"] = pd.to_numeric(paired_area["NDVI_S2"], errors="coerce")
paired_area["NDVI_L89"] = pd.to_numeric(paired_area["NDVI_L89"], errors="coerce")

ground = ground.dropna(subset=["Plot", "Landuse", "LAI"]).copy()

s2_area = s2_area.dropna(subset=["Plot", "Landuse", "NDVI_S2"]).copy()
s2_pixel = s2_pixel.dropna(subset=["Plot", "Landuse", "NDVI_S2"]).copy()

l89_area = l89_area.dropna(subset=["Plot", "Landuse", "NDVI_L89"]).copy()
l89_pixel = l89_pixel.dropna(subset=["Plot", "Landuse", "NDVI_L89"]).copy()

paired_area = paired_area.dropna(subset=["Plot", "Landuse", "LAI", "NDVI_S2", "NDVI_L89"]).copy()

# Keep only target land uses
for name in ["ground", "s2_area", "s2_pixel", "l89_area", "l89_pixel", "paired_area"]:
    locals()[name] = locals()[name][locals()[name]["Landuse"].isin(["Forest", "Oil palm"])].copy()


# Fit model and calculate absolute error

def fit_model_and_error(df_sensor, ground_df, x_col, sensor_name, method_name):
    """
    Fit LAI ~ NDVI separately by land use.
    Return dataframe with predicted LAI and absolute error.
    """
    merged = ground_df[["Plot", "Landuse", "LAI"]].merge(
        df_sensor[["Plot", "Landuse", x_col]],
        on=["Plot", "Landuse"],
        how="inner"
    )

    out_rows = []

    for lc in ["Forest", "Oil palm"]:
        sub = merged[merged["Landuse"] == lc].dropna(subset=["LAI", x_col]).copy()

        if len(sub) < 2:
            continue

        x = sub[x_col].values
        y = sub["LAI"].values

        slope, intercept, r, p, se = linregress(x, y)
        yhat = slope * x + intercept
        abs_error = np.abs(yhat - y)

        tmp = sub.copy()
        tmp["Sensor"] = sensor_name
        tmp["Method"] = method_name
        tmp["Predictor"] = x_col
        tmp["LAI_pred"] = yhat
        tmp["abs_error"] = abs_error
        tmp["slope"] = slope
        tmp["intercept"] = intercept
        tmp["R2_fit"] = r**2

        out_rows.append(tmp)

    if len(out_rows) == 0:
        return pd.DataFrame()

    return pd.concat(out_rows, ignore_index=True)

def fit_model_and_error_on_subset(df_subset, x_col, sensor_name, method_name):
    """
    Fit LAI ~ NDVI on exact paired subset.
    """
    out_rows = []

    for lc in ["Forest", "Oil palm"]:
        sub = df_subset[df_subset["Landuse"] == lc].dropna(subset=["LAI", x_col]).copy()

        if len(sub) < 2:
            continue

        x = sub[x_col].values
        y = sub["LAI"].values

        slope, intercept, r, p, se = linregress(x, y)
        yhat = slope * x + intercept
        abs_error = np.abs(yhat - y)

        tmp = sub.copy()
        tmp["Sensor"] = sensor_name
        tmp["Method"] = method_name
        tmp["Predictor"] = x_col
        tmp["LAI_pred"] = yhat
        tmp["abs_error"] = abs_error
        tmp["slope"] = slope
        tmp["intercept"] = intercept
        tmp["R2_fit"] = r**2

        out_rows.append(tmp)

    if len(out_rows) == 0:
        return pd.DataFrame()

    return pd.concat(out_rows, ignore_index=True)

def paired_error_test(err_s2, err_l89, method_name):
    """
    Paired t-test using exact same plots.
    """
    rows = []

    for lc in ["Forest", "Oil palm"]:
        s2_sub = err_s2[err_s2["Landuse"] == lc][["Plot", "Landuse", "abs_error"]].rename(columns={"abs_error": "abs_error_S2"})
        l89_sub = err_l89[err_l89["Landuse"] == lc][["Plot", "Landuse", "abs_error"]].rename(columns={"abs_error": "abs_error_L89"})

        pair = s2_sub.merge(l89_sub, on=["Plot", "Landuse"], how="inner")

        n = len(pair)

        if n >= 2:
            t_stat, p_val = ttest_rel(pair["abs_error_S2"], pair["abs_error_L89"])
            mae_s2 = pair["abs_error_S2"].mean()
            mae_l89 = pair["abs_error_L89"].mean()
        else:
            t_stat, p_val, mae_s2, mae_l89 = np.nan, np.nan, np.nan, np.nan

        rows.append({
            "Land use": lc,
            "Method": method_name,
            "Test type": "Paired t-test",
            "n": n,
            "Mean abs. error S2": mae_s2,
            "Mean abs. error L89": mae_l89,
            "t-statistic": t_stat,
            "p-value": p_val
        })

    return pd.DataFrame(rows)

def unpaired_error_test(err_s2, err_l89, method_name):
    """
    Welch unpaired t-test using full valid samples.
    """
    rows = []

    for lc in ["Forest", "Oil palm"]:
        s2_sub = err_s2[err_s2["Landuse"] == lc]["abs_error"].dropna().values
        l89_sub = err_l89[err_l89["Landuse"] == lc]["abs_error"].dropna().values

        n = min(len(s2_sub), len(l89_sub))
        n_s2 = len(s2_sub)
        n_l89 = len(l89_sub)

        if n_s2 >= 2 and n_l89 >= 2:
            t_stat, p_val = ttest_ind(s2_sub, l89_sub, equal_var=False)
            mae_s2 = np.mean(s2_sub)
            mae_l89 = np.mean(l89_sub)
        else:
            t_stat, p_val, mae_s2, mae_l89 = np.nan, np.nan, np.nan, np.nan

        rows.append({
            "Land use": lc,
            "Method": method_name,
            "Test type": "Welch unpaired t-test",
            "n": f"S2={n_s2}, L89={n_l89}",
            "Mean abs. error S2": mae_s2,
            "Mean abs. error L89": mae_l89,
            "t-statistic": t_stat,
            "p-value": p_val
        })

    return pd.DataFrame(rows)


# Build error tables

# Full valid samples
err_s2_area  = fit_model_and_error(s2_area, ground, "NDVI_S2", "Sentinel-2", "Area")
err_s2_pixel = fit_model_and_error(s2_pixel, ground, "NDVI_S2", "Sentinel-2", "Pixel")

err_l89_area  = fit_model_and_error(l89_area, ground, "NDVI_L89", "Landsat 8/9", "Area")
err_l89_pixel = fit_model_and_error(l89_pixel, ground, "NDVI_L89", "Landsat 8/9", "Pixel")

# Exact paired area-weighted subset
err_s2_area_paired  = fit_model_and_error_on_subset(paired_area, "NDVI_S2", "Sentinel-2", "Area")
err_l89_area_paired = fit_model_and_error_on_subset(paired_area, "NDVI_L89", "Landsat 8/9", "Area")


# Run tests

results = []

# Area-weighted: paired + unpaired
results.append(paired_error_test(err_s2_area_paired, err_l89_area_paired, "Area"))
results.append(unpaired_error_test(err_s2_area, err_l89_area, "Area"))

# Single-pixel: unpaired only
results.append(unpaired_error_test(err_s2_pixel, err_l89_pixel, "Pixel"))

results_table = pd.concat(results, ignore_index=True)


# Round and print

for col in ["Mean abs. error S2", "Mean abs. error L89", "t-statistic", "p-value"]:
    results_table[col] = pd.to_numeric(results_table[col], errors="coerce").round(3)


print(results_table.to_string(index=False))


# Save outputs

results_path = os.path.join(BASE, "sensor_error_ttests_summary.csv")
results_table.to_csv(results_path, index=False)

# Save fitted error tables
err_s2_area.to_csv(os.path.join(BASE, "S2_area_absolute_errors.csv"), index=False)
err_l89_area.to_csv(os.path.join(BASE, "L89_area_absolute_errors.csv"), index=False)
err_s2_pixel.to_csv(os.path.join(BASE, "S2_pixel_absolute_errors.csv"), index=False)
err_l89_pixel.to_csv(os.path.join(BASE, "L89_pixel_absolute_errors.csv"), index=False)



In [ ]:
# Area vs pixel t-tests for all Sentinel-2 indices

# Processing workflow:
#  Load Sentinel-2 single-pixel and area-weighted datasets
#  Fit separate LAI ~ index models by land use
#  Calculate absolute prediction error for each method
#  Compare area and pixel methods using:
#       Paired t-test on overlapping plots
#       Welch unpaired t-test on full valid samples
#
# Outputs:
#   Mean abs. error Pixel = mean absolute error for single-pixel method
#   Mean abs. error Area = mean absolute error for area-weighted method
#   t-statistic = t-test statistic
#   p-value = test significance


from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
from scipy.stats import linregress, ttest_rel, ttest_ind

BASE = "/content/drive/MyDrive/LAI/"


# Load datasets

pixel = pd.read_csv(os.path.join(BASE, "S2_STRICT_singlepixel_rededge.csv"))
area  = pd.read_csv(os.path.join(BASE, "S2_STRICT_weighted_1000m2.csv"))


# Clean data

for df in [pixel, area]:
    df["Plot"] = df["Plot"].astype(str).str.strip()
    df["Landuse"] = df["Landuse"].astype(str).str.strip()
    df["LAI"] = pd.to_numeric(df["LAI"], errors="coerce")

pixel_cols = ["NDVI_pixel", "NDRE705_pixel", "CIre705_pixel", "MTCI_pixel"]
area_cols  = ["NDVI_weighted", "NDRE705_weighted", "CIre705_weighted", "MTCI_weighted"]

for col in pixel_cols:
    if col in pixel.columns:
        pixel[col] = pd.to_numeric(pixel[col], errors="coerce")

for col in area_cols:
    if col in area.columns:
        area[col] = pd.to_numeric(area[col], errors="coerce")

pixel = pixel[pixel["Landuse"].isin(["Forest", "Oil palm"])].copy()
area  = area[area["Landuse"].isin(["Forest", "Oil palm"])].copy()


# Fit model and calculate absolute error

def fit_errors(df, x_col, method_name, index_name):
    """
    Fit LAI - index separately by land use.
    Return dataframe with absolute prediction error.
    """
    rows = []

    for lc in ["Forest", "Oil palm"]:
        sub = df[df["Landuse"] == lc].dropna(subset=["Plot", "LAI", x_col]).copy()

        if len(sub) < 2:
            continue

        x = sub[x_col].values
        y = sub["LAI"].values

        slope, intercept, r, p, se = linregress(x, y)
        yhat = slope * x + intercept
        abs_error = np.abs(yhat - y)

        tmp = sub[["Plot", "Landuse", "LAI", x_col]].copy()
        tmp["Index"] = index_name
        tmp["Method"] = method_name
        tmp["Predicted_LAI"] = yhat
        tmp["Abs_error"] = abs_error
        tmp["R2_fit"] = r**2
        tmp["Slope"] = slope
        tmp["Intercept"] = intercept

        rows.append(tmp)

    if len(rows) == 0:
        return pd.DataFrame()

    return pd.concat(rows, ignore_index=True)

def paired_test(err_pixel, err_area, index_name):
    """
    Paired t-test on overlapping plots between methods.
    """
    rows = []

    for lc in ["Forest", "Oil palm"]:
        p = err_pixel[err_pixel["Landuse"] == lc][["Plot", "Landuse", "Abs_error"]].rename(
            columns={"Abs_error": "Abs_error_pixel"}
        )
        a = err_area[err_area["Landuse"] == lc][["Plot", "Landuse", "Abs_error"]].rename(
            columns={"Abs_error": "Abs_error_area"}
        )

        pair = p.merge(a, on=["Plot", "Landuse"], how="inner")
        n = len(pair)

        if n >= 2:
            t_stat, p_val = ttest_rel(pair["Abs_error_pixel"], pair["Abs_error_area"])
            mae_pixel = pair["Abs_error_pixel"].mean()
            mae_area = pair["Abs_error_area"].mean()
        else:
            t_stat, p_val, mae_pixel, mae_area = np.nan, np.nan, np.nan, np.nan

        rows.append({
            "Index": index_name,
            "Land use": lc,
            "Comparison": "Area vs Pixel",
            "Test type": "Paired t-test",
            "n": n,
            "Mean abs. error Pixel": mae_pixel,
            "Mean abs. error Area": mae_area,
            "t-statistic": t_stat,
            "p-value": p_val
        })

    return pd.DataFrame(rows)

def unpaired_test(err_pixel, err_area, index_name):
    """
    Welch unpaired t-test on full valid samples.
    """
    rows = []

    for lc in ["Forest", "Oil palm"]:
        p = err_pixel[err_pixel["Landuse"] == lc]["Abs_error"].dropna().values
        a = err_area[err_area["Landuse"] == lc]["Abs_error"].dropna().values

        n_p = len(p)
        n_a = len(a)

        if n_p >= 2 and n_a >= 2:
            t_stat, p_val = ttest_ind(p, a, equal_var=False)
            mae_pixel = np.mean(p)
            mae_area = np.mean(a)
        else:
            t_stat, p_val, mae_pixel, mae_area = np.nan, np.nan, np.nan, np.nan

        rows.append({
            "Index": index_name,
            "Land use": lc,
            "Comparison": "Area vs Pixel",
            "Test type": "Welch unpaired t-test",
            "n": f"Pixel={n_p}, Area={n_a}",
            "Mean abs. error Pixel": mae_pixel,
            "Mean abs. error Area": mae_area,
            "t-statistic": t_stat,
            "p-value": p_val
        })

    return pd.DataFrame(rows)


# Define index mapping

index_map = [
    ("NDVI", "NDVI_pixel", "NDVI_weighted"),
    ("NDRE705", "NDRE705_pixel", "NDRE705_weighted"),
    ("CIre705", "CIre705_pixel", "CIre705_weighted"),
    ("MTCI", "MTCI_pixel", "MTCI_weighted"),
]


# Run all tests

all_results = []
all_error_tables = []

for index_name, pixel_col, area_col in index_map:
    err_pixel = fit_errors(pixel, pixel_col, "Pixel", index_name)
    err_area  = fit_errors(area, area_col, "Area", index_name)

    all_error_tables.append(err_pixel)
    all_error_tables.append(err_area)

    all_results.append(paired_test(err_pixel, err_area, index_name))
    all_results.append(unpaired_test(err_pixel, err_area, index_name))

results_table = pd.concat(all_results, ignore_index=True)


# Sort and round

index_order = {"NDVI": 0, "NDRE705": 1, "CIre705": 2, "MTCI": 3}
landuse_order = {"Forest": 0, "Oil palm": 1}
test_order = {"Paired t-test": 0, "Welch unpaired t-test": 1}

results_table["_index"] = results_table["Index"].map(index_order)
results_table["_landuse"] = results_table["Land use"].map(landuse_order)
results_table["_test"] = results_table["Test type"].map(test_order)

results_table = results_table.sort_values(
    ["_index", "_landuse", "_test"]
).drop(columns=["_index", "_landuse", "_test"])

for col in ["Mean abs. error Pixel", "Mean abs. error Area", "t-statistic", "p-value"]:
    results_table[col] = pd.to_numeric(results_table[col], errors="coerce").round(3)


print(results_table.to_string(index=False))


# Save outputs

results_path = os.path.join(BASE, "area_vs_pixel_ttests_all_indices.csv")
results_table.to_csv(results_path, index=False)

# Save fitted error tables
errors_combined = pd.concat(all_error_tables, ignore_index=True)
errors_path = os.path.join(BASE, "area_vs_pixel_absolute_errors_all_indices.csv")
errors_combined.to_csv(errors_path, index=False)



In [ ]:
# Landsat 8/9 NDVI area vs pixel t-test

# Processing workflow:
#  Load ground, Landsat single-pixel, and Landsat area-weighted datasets
#  Merge Landsat datasets with ground LAI
#  Fit separate LAI ~ NDVI models by land use
#  Calculate absolute prediction error for each method
#  Compare area and pixel methods using:
#       Paired t-test on overlapping plots
#       Welch unpaired t-test on full valid samples
#
# Outputs:
#   Mean abs. error Pixel = mean absolute error for single-pixel method
#   Mean abs. error Area = mean absolute error for area-weighted method
#   t-statistic = t-test statistic
#   p-value = test significance


import os
import numpy as np
import pandas as pd
from scipy.stats import linregress, ttest_rel, ttest_ind

BASE = "/content/drive/MyDrive/LAI/"


# Load datasets

ground = pd.read_csv(os.path.join(BASE, "LAI_GROUND.csv"))
pixel  = pd.read_csv(os.path.join(BASE, "Copy of L89_singlepixel_valid_plots_used.csv"))
area   = pd.read_csv(os.path.join(BASE, "L89_valid_plots_used.csv"))


# Clean data

for df in [ground, pixel, area]:
    df["Plot"] = df["Plot"].astype(str).str.strip()
    df["Landuse"] = df["Landuse"].astype(str).str.strip()

ground["LAI"] = pd.to_numeric(ground["LAI"], errors="coerce")
pixel["NDVI_L89"] = pd.to_numeric(pixel["NDVI_L89"], errors="coerce")
area["NDVI_L89"] = pd.to_numeric(area["NDVI_L89"], errors="coerce")

ground = ground[ground["Landuse"].isin(["Forest", "Oil palm"])].copy()
pixel  = pixel[pixel["Landuse"].isin(["Forest", "Oil palm"])].copy()
area   = area[area["Landuse"].isin(["Forest", "Oil palm"])].copy()

ground = ground.dropna(subset=["Plot", "Landuse", "LAI"]).copy()
pixel  = pixel.dropna(subset=["Plot", "Landuse", "NDVI_L89"]).copy()
area   = area.dropna(subset=["Plot", "Landuse", "NDVI_L89"]).copy()


# Merge Landsat datasets with ground LAI

pixel = ground[["Plot", "Landuse", "LAI"]].merge(
    pixel,
    on=["Plot", "Landuse"],
    how="inner"
)

area = ground[["Plot", "Landuse", "LAI"]].merge(
    area,
    on=["Plot", "Landuse"],
    how="inner"
)

print("Merged pixel rows:", len(pixel))
print("Merged area rows:", len(area))


# Fit model and calculate absolute error

def fit_errors(df, x_col, method_name):
    rows = []

    for lc in ["Forest", "Oil palm"]:
        sub = df[df["Landuse"] == lc].dropna(subset=["Plot", "LAI", x_col]).copy()

        if len(sub) < 2:
            continue

        x = sub[x_col].values
        y = sub["LAI"].values

        slope, intercept, r, p, se = linregress(x, y)
        yhat = slope * x + intercept
        abs_error = np.abs(yhat - y)

        tmp = sub[["Plot", "Landuse", "LAI", x_col]].copy()
        tmp["Method"] = method_name
        tmp["Predicted_LAI"] = yhat
        tmp["Abs_error"] = abs_error
        tmp["R2_fit"] = r**2
        tmp["Slope"] = slope
        tmp["Intercept"] = intercept

        rows.append(tmp)

    if len(rows) == 0:
        return pd.DataFrame()

    return pd.concat(rows, ignore_index=True)


# Fit separate models

err_pixel = fit_errors(pixel, "NDVI_L89", "Pixel")
err_area  = fit_errors(area, "NDVI_L89", "Area")


# Paired t-test

paired_rows = []

for lc in ["Forest", "Oil palm"]:
    p = err_pixel[err_pixel["Landuse"] == lc][["Plot", "Landuse", "Abs_error"]].rename(
        columns={"Abs_error": "Abs_error_pixel"}
    )
    a = err_area[err_area["Landuse"] == lc][["Plot", "Landuse", "Abs_error"]].rename(
        columns={"Abs_error": "Abs_error_area"}
    )

    pair = p.merge(a, on=["Plot", "Landuse"], how="inner")
    n = len(pair)

    if n >= 2:
        t_stat, p_val = ttest_rel(pair["Abs_error_pixel"], pair["Abs_error_area"])
        mae_pixel = pair["Abs_error_pixel"].mean()
        mae_area = pair["Abs_error_area"].mean()
    else:
        t_stat, p_val, mae_pixel, mae_area = np.nan, np.nan, np.nan, np.nan

    paired_rows.append({
        "Index": "NDVI",
        "Land use": lc,
        "Comparison": "Area vs Pixel",
        "Test type": "Paired t-test",
        "n": n,
        "Mean abs. error Pixel": mae_pixel,
        "Mean abs. error Area": mae_area,
        "t-statistic": t_stat,
        "p-value": p_val
    })

paired_table = pd.DataFrame(paired_rows)


# Welch unpaired t-test

unpaired_rows = []

for lc in ["Forest", "Oil palm"]:
    p = err_pixel[err_pixel["Landuse"] == lc]["Abs_error"].dropna().values
    a = err_area[err_area["Landuse"] == lc]["Abs_error"].dropna().values

    n_p = len(p)
    n_a = len(a)

    if n_p >= 2 and n_a >= 2:
        t_stat, p_val = ttest_ind(p, a, equal_var=False)
        mae_pixel = np.mean(p)
        mae_area = np.mean(a)
    else:
        t_stat, p_val, mae_pixel, mae_area = np.nan, np.nan, np.nan, np.nan

    unpaired_rows.append({
        "Index": "NDVI",
        "Land use": lc,
        "Comparison": "Area vs Pixel",
        "Test type": "Welch unpaired t-test",
        "n": f"Pixel={n_p}, Area={n_a}",
        "Mean abs. error Pixel": mae_pixel,
        "Mean abs. error Area": mae_area,
        "t-statistic": t_stat,
        "p-value": p_val
    })

unpaired_table = pd.DataFrame(unpaired_rows)


# Combine and round

results_table = pd.concat([paired_table, unpaired_table], ignore_index=True)

test_order = {"Paired t-test": 0, "Welch unpaired t-test": 1}
landuse_order = {"Forest": 0, "Oil palm": 1}

results_table["_test"] = results_table["Test type"].map(test_order)
results_table["_landuse"] = results_table["Land use"].map(landuse_order)

results_table = results_table.sort_values(
    ["_landuse", "_test"]
).drop(columns=["_test", "_landuse"])

for col in ["Mean abs. error Pixel", "Mean abs. error Area", "t-statistic", "p-value"]:
    results_table[col] = pd.to_numeric(results_table[col], errors="coerce").round(3)


print(results_table.to_string(index=False))


# Save outputs

results_path = os.path.join(BASE, "L89_NDVI_area_vs_pixel_ttests.csv")
results_table.to_csv(results_path, index=False)

errors_combined = pd.concat([err_pixel, err_area], ignore_index=True)
errors_path = os.path.join(BASE, "L89_NDVI_area_vs_pixel_absolute_errors.csv")
errors_combined.to_csv(errors_path, index=False)


# Figures

# ndvi snapshots

In [ ]:
# sentinel-2 ndvi snapshots (150 m squares)

# processing workflow:
# load plot data and dates
# build 150 x 150 m square per plot
# select best image within time window
# export ndvi (tif + png)




import os
import ee
import geemap
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable


# initialise earth engine
try:
    ee.Initialize(project="double-skyline-471010-g2")
except:
    ee.Authenticate()
    ee.Initialize(project="double-skyline-471010-g2")


# paths
BASE = "/content/drive/MyDrive/LAI/"
OUT_DIR = os.path.join(BASE, "FIG_ndvi_snapshots_S2_150m_square")
os.makedirs(OUT_DIR, exist_ok=True)


# load data
INFILE = os.path.join(BASE, "LAI_GROUND.csv")
df = pd.read_csv(INFILE)

df["Plot"] = df["Plot"].astype(str).str.strip()
df["Landuse"] = df["Landuse"].astype(str).str.strip()
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

# use date_centr
if "date_center" in df.columns:
    df["date_center"] = pd.to_datetime(df["date_center"], errors="coerce")
else:
    df["date_center"] = pd.to_datetime(df["Date"], errors="coerce")

df = df.dropna(subset=["Plot", "lat", "lon", "date_center"]).copy()


# select plots
forest_plots = ["BF2", "BF3", "BF4", "F06", "F13", "F16"]
palm_plots   = ["HO3", "O03", "O14", "O16", "O17", "O18"]
target_plots = forest_plots + palm_plots

snap_df = df[df["Plot"].isin(target_plots)].copy()

print("plots found:")
print(snap_df[["Plot", "Landuse", "date_center"]].sort_values(["Landuse", "Plot"]).to_string(index=False))


# parameters
EXPORT_SIDE_M = 150.0
EXPORT_HALF_M = EXPORT_SIDE_M / 2.0

MAX_SCENE_CLOUD_PCT = 80
CLOUD_PROB_THR = 40
DILATE_METERS = 60
WINDOWS = [15, 30, 50]

MIN_VALID_FRACTION = 0.80

NDVI_MIN = 0.65
NDVI_MAX = 0.95
PALETTE = ["#ffffcc", "#c2e699", "#78c679", "#31a354", "#006837"]

EXPORT_SCALE_M = 10
PNG_FIGSIZE = 6
PNG_DPI = 300


# sentinel-2 collections
S2_SR = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
S2_CPROB = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")

join = ee.Join.saveFirst("cloud_prob_img")
join_filter = ee.Filter.equals(leftField="system:index", rightField="system:index")

s2_joined = join.apply(
    primary=S2_SR,
    secondary=S2_CPROB,
    condition=join_filter
)


# helper functions

def safe_name(s):
    return str(s).replace(" ", "_").replace("/", "_")


def strict_mask_s2(img):
    # apply strict cloud + shadow mask with dilation
    scl = img.select("SCL")

    good = (
        scl.neq(0)
        .And(scl.neq(1))
        .And(scl.neq(2))
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    qa = img.select("QA60")
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11

    qa_clear = qa.bitwiseAnd(cloud_bit).eq(0).And(
        qa.bitwiseAnd(cirrus_bit).eq(0)
    )

    cprob = ee.Image(img.get("cloud_prob_img")).select("probability")
    cprob_clear = cprob.lt(CLOUD_PROB_THR)

    mask = good.And(qa_clear).And(cprob_clear)

    if DILATE_METERS > 0:
        bad = mask.Not()
        bad = bad.focal_max(radius=DILATE_METERS, units="meters")
        mask = bad.Not()

    return img.updateMask(mask)


def add_ndvi(img):
    # compute ndvi
    red = img.select("B4").multiply(0.0001)
    nir = img.select("B8").multiply(0.0001)
    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI_S2")
    return img.addBands(ndvi).copyProperties(
        img, ["system:time_start", "CLOUDY_PIXEL_PERCENTAGE"]
    )


def build_export_feature(row):
    # create 150 m square around plot
    lon = float(row["lon"])
    lat = float(row["lat"])
    point = ee.Geometry.Point([lon, lat])
    square = point.buffer(EXPORT_HALF_M).bounds()

    return ee.Feature(square, {
        "Plot": str(row["Plot"]),
        "Landuse": str(row["Landuse"]),
        "date_center": pd.to_datetime(row["date_center"]).strftime("%Y-%m-%d")
    })


def best_s2_image_for_plot(feat):
    # select image with highest valid area and closest date
    geom = ee.Feature(feat).geometry()
    center = ee.Date(ee.Feature(feat).get("date_center"))
    full_area = ee.Number(EXPORT_SIDE_M * EXPORT_SIDE_M)

    def collection_for(days):
        return (
            ee.ImageCollection(s2_joined)
            .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", MAX_SCENE_CLOUD_PCT))
            .filterBounds(geom)
            .filterDate(center.advance(-days, "day"), center.advance(days, "day"))
            .map(strict_mask_s2)
            .map(add_ndvi)
            .select(["NDVI_S2"])
        )

    def add_valid_area(img):
        # calculate valid area and fraction
        valid_area = (
            ee.Image.pixelArea()
            .rename("pxarea")
            .updateMask(img.select("NDVI_S2").mask())
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=geom,
                scale=10,
                maxPixels=1e7
            )
            .get("pxarea")
        )

        valid_area = ee.Number(ee.Algorithms.If(valid_area, valid_area, 0))
        valid_frac = valid_area.divide(full_area)
        abs_days = ee.Number(img.date().difference(center, "day")).abs()

        return img.set({
            "valid_area_m2": valid_area,
            "valid_frac": valid_frac,
            "abs_days": abs_days
        })

    for days in WINDOWS:
        ic = collection_for(days).map(add_valid_area)

        ic = ic.filter(ee.Filter.gte("valid_frac", MIN_VALID_FRACTION))
        ic = ic.sort("valid_frac", False).sort("abs_days").sort("CLOUDY_PIXEL_PERCENTAGE")

        if ic.size().getInfo() > 0:
            return ee.Image(ic.first())

    return None


def tif_to_large_png_rgba(tif_path, png_path, fig_inches=6, dpi=300):
    # convert tif to clean png
    with rasterio.open(tif_path) as src:
        arr = src.read()

    rgb = np.transpose(arr[:3], (1, 2, 0)).astype(np.uint8)

    if arr.shape[0] >= 4:
        alpha = arr[3].astype(np.uint8)
    else:
        alpha = np.where(np.any(rgb > 0, axis=2), 255, 0).astype(np.uint8)

    rgba = np.dstack([rgb, alpha])

    fig, ax = plt.subplots(figsize=(fig_inches, fig_inches), dpi=dpi)
    ax.imshow(rgba, interpolation="nearest")
    ax.axis("off")
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    plt.savefig(png_path, dpi=dpi, bbox_inches="tight", pad_inches=0, transparent=True)
    plt.close(fig)


# export snapshots
exported = []

for _, row in snap_df.sort_values(["Landuse", "Plot"]).iterrows():
    feat = build_export_feature(row)
    plot_id = str(row["Plot"])
    landuse = str(row["Landuse"])


    img = best_s2_image_for_plot(feat)

    if img is None:
        continue

    date_str = ee.Date(img.get("system:time_start")).format("YYYY-MM-dd").getInfo()
    geom = ee.Feature(feat).geometry()
    valid_area = ee.Number(img.get("valid_area_m2")).getInfo()
    valid_frac = ee.Number(img.get("valid_frac")).getInfo()

    ndvi = img.select("NDVI_S2").clip(geom)
    ndvi_vis = ndvi.visualize(min=NDVI_MIN, max=NDVI_MAX, palette=PALETTE)

    tif_path = os.path.join(OUT_DIR, f"{safe_name(plot_id)}_{date_str}_S2_NDVI_150mSquare.tif")
    png_path = os.path.join(OUT_DIR, f"{safe_name(plot_id)}_{date_str}_S2_NDVI_150mSquare.png")

    geemap.ee_export_image(
        ndvi_vis,
        filename=tif_path,
        scale=EXPORT_SCALE_M,
        region=geom.bounds(),
        file_per_band=False
    )

    tif_to_large_png_rgba(tif_path, png_path, fig_inches=PNG_FIGSIZE, dpi=PNG_DPI)

    exported.append({
        "Plot": plot_id,
        "Landuse": landuse,
        "Sensor": "Sentinel-2",
        "Date": date_str,
        "Valid_area_m2": valid_area,
        "Valid_fraction": valid_frac,
        "TIFF": tif_path,
        "PNG": png_path
    })


# save log
log_df = pd.DataFrame(exported)
log_path = os.path.join(OUT_DIR, "S2_snapshot_export_log.csv")
log_df.to_csv(log_path, index=False)




# save legend
cmap = LinearSegmentedColormap.from_list("ndvi_yellow_green", PALETTE, N=256)
norm = Normalize(vmin=NDVI_MIN, vmax=NDVI_MAX)

fig = plt.figure(figsize=(1.8, 5.0))
ax = fig.add_axes([0.35, 0.08, 0.30, 0.84])

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = plt.colorbar(sm, cax=ax)
cbar.set_label("NDVI", rotation=90)
cbar.set_ticks([0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95])

legend_png = os.path.join(OUT_DIR, "S2_NDVI_legend_0p65_to_0p95.png")
plt.savefig(legend_png, dpi=400, bbox_inches="tight", transparent=True)
plt.close(fig)



In [ ]:
# sentinel-2 ndvi snapshots for 150 m squares

# processing workflow:
# load plot data
# select plots
# build 150 x 150 m square per plot





import os
import ee
import geemap
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable


# initialise earth engine
try:
    ee.Initialize(project="double-skyline-471010-g2")
except:
    ee.Authenticate()
    ee.Initialize(project="double-skyline-471010-g2")


# paths
BASE = "/content/drive/MyDrive/FINAL_ONE/"
OUT_DIR = os.path.join(BASE, "FIG_ndvi_snapshots_S2_150m_square")
os.makedirs(OUT_DIR, exist_ok=True)


# load data
INFILE = os.path.join(BASE, "LAI_GROUND.csv")
df = pd.read_csv(INFILE)

df["Plot"] = df["Plot"].astype(str).str.strip()
df["Landuse"] = df["Landuse"].astype(str).str.strip()
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

# use date_centre
if "date_center" in df.columns:
    df["date_center"] = pd.to_datetime(df["date_center"], errors="coerce")
else:
    df["date_center"] = pd.to_datetime(df["Date"], errors="coerce")

df = df.dropna(subset=["Plot", "lat", "lon", "date_center"]).copy()


# select plots
forest_plots = ["BF2", "BF3", "BF4", "F06", "F13", "F16"]
palm_plots   = ["HO3", "O03", "O14", "O16", "O17", "O18"]
target_plots = forest_plots + palm_plots

snap_df = df[df["Plot"].isin(target_plots)].copy()

print("plots found:")
print(snap_df[["Plot", "Landuse", "date_center"]].sort_values(["Landuse", "Plot"]).to_string(index=False))


# parameters
EXPORT_SIDE_M = 150.0
EXPORT_HALF_M = EXPORT_SIDE_M / 2.0

MAX_SCENE_CLOUD_PCT = 80
CLOUD_PROB_THR = 40
DILATE_METERS = 60
WINDOWS = [15, 30, 50]

MIN_VALID_FRACTION = 0.80

NDVI_MIN = 0.65
NDVI_MAX = 0.95
PALETTE = ["#ffffcc", "#c2e699", "#78c679", "#31a354", "#006837"]

EXPORT_SCALE_M = 10
PNG_FIGSIZE = 6
PNG_DPI = 300


# load sentinel-2 collections
S2_SR = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
S2_CPROB = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")

join = ee.Join.saveFirst("cloud_prob_img")
join_filter = ee.Filter.equals(leftField="system:index", rightField="system:index")

s2_joined = join.apply(
    primary=S2_SR,
    secondary=S2_CPROB,
    condition=join_filter
)


# helper functions

def safe_name(s):
    return str(s).replace(" ", "_").replace("/", "_")


def strict_mask_s2(img):
    # apply strict mask
    scl = img.select("SCL")

    good = (
        scl.neq(0)
        .And(scl.neq(1))
        .And(scl.neq(2))
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    qa = img.select("QA60")
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11

    qa_clear = qa.bitwiseAnd(cloud_bit).eq(0).And(
        qa.bitwiseAnd(cirrus_bit).eq(0)
    )

    cprob = ee.Image(img.get("cloud_prob_img")).select("probability")
    cprob_clear = cprob.lt(CLOUD_PROB_THR)

    mask = good.And(qa_clear).And(cprob_clear)

    if DILATE_METERS > 0:
        bad = mask.Not()
        bad = bad.focal_max(radius=DILATE_METERS, units="meters")
        mask = bad.Not()

    return img.updateMask(mask)


def add_ndvi(img):
    # calculate ndvi
    red = img.select("B4").multiply(0.0001)
    nir = img.select("B8").multiply(0.0001)
    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI_S2")
    return img.addBands(ndvi).copyProperties(
        img, ["system:time_start", "CLOUDY_PIXEL_PERCENTAGE"]
    )


def build_export_feature(row):
    # build square around plot
    lon = float(row["lon"])
    lat = float(row["lat"])
    point = ee.Geometry.Point([lon, lat])
    square = point.buffer(EXPORT_HALF_M).bounds()

    return ee.Feature(square, {
        "Plot": str(row["Plot"]),
        "Landuse": str(row["Landuse"]),
        "date_center": pd.to_datetime(row["date_center"]).strftime("%Y-%m-%d")
    })


def best_s2_image_for_plot(feat):
    # find best image for each plot
    geom = ee.Feature(feat).geometry()
    center = ee.Date(ee.Feature(feat).get("date_center"))
    full_area = ee.Number(EXPORT_SIDE_M * EXPORT_SIDE_M)

    def collection_for(days):
        return (
            ee.ImageCollection(s2_joined)
            .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", MAX_SCENE_CLOUD_PCT))
            .filterBounds(geom)
            .filterDate(center.advance(-days, "day"), center.advance(days, "day"))
            .map(strict_mask_s2)
            .map(add_ndvi)
            .select(["NDVI_S2"])
        )

    def add_valid_area(img):
        # calculate valid area
        valid_area = (
            ee.Image.pixelArea()
            .rename("pxarea")
            .updateMask(img.select("NDVI_S2").mask())
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=geom,
                scale=10,
                maxPixels=1e7
            )
            .get("pxarea")
        )

        valid_area = ee.Number(ee.Algorithms.If(valid_area, valid_area, 0))
        valid_frac = valid_area.divide(full_area)
        abs_days = ee.Number(img.date().difference(center, "day")).abs()

        return img.set({
            "valid_area_m2": valid_area,
            "valid_frac": valid_frac,
            "abs_days": abs_days
        })

    for days in WINDOWS:
        ic = collection_for(days).map(add_valid_area)

        ic = ic.filter(ee.Filter.gte("valid_frac", MIN_VALID_FRACTION))
        ic = ic.sort("valid_frac", False).sort("abs_days").sort("CLOUDY_PIXEL_PERCENTAGE")

        if ic.size().getInfo() > 0:
            return ee.Image(ic.first())

    return None


def tif_to_large_png_rgba(tif_path, png_path, fig_inches=6, dpi=300):
    # convert tif to png
    with rasterio.open(tif_path) as src:
        arr = src.read()

    rgb = np.transpose(arr[:3], (1, 2, 0)).astype(np.uint8)

    if arr.shape[0] >= 4:
        alpha = arr[3].astype(np.uint8)
    else:
        alpha = np.where(np.any(rgb > 0, axis=2), 255, 0).astype(np.uint8)

    rgba = np.dstack([rgb, alpha])

    fig, ax = plt.subplots(figsize=(fig_inches, fig_inches), dpi=dpi)
    ax.imshow(rgba, interpolation="nearest")
    ax.axis("off")
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    plt.savefig(png_path, dpi=dpi, bbox_inches="tight", pad_inches=0, transparent=True)
    plt.close(fig)


# export snapshots
exported = []

for _, row in snap_df.sort_values(["Landuse", "Plot"]).iterrows():
    feat = build_export_feature(row)
    plot_id = str(row["Plot"])
    landuse = str(row["Landuse"])

    print(f"\nprocessing sentinel-2: {plot_id} ({landuse})")

    img = best_s2_image_for_plot(feat)

    if img is None:
        continue

    date_str = ee.Date(img.get("system:time_start")).format("YYYY-MM-dd").getInfo()
    geom = ee.Feature(feat).geometry()
    valid_area = ee.Number(img.get("valid_area_m2")).getInfo()
    valid_frac = ee.Number(img.get("valid_frac")).getInfo()

    ndvi = img.select("NDVI_S2").clip(geom)
    ndvi_vis = ndvi.visualize(min=NDVI_MIN, max=NDVI_MAX, palette=PALETTE)

    tif_path = os.path.join(OUT_DIR, f"{safe_name(plot_id)}_{date_str}_S2_NDVI_150mSquare.tif")
    png_path = os.path.join(OUT_DIR, f"{safe_name(plot_id)}_{date_str}_S2_NDVI_150mSquare.png")

    geemap.ee_export_image(
        ndvi_vis,
        filename=tif_path,
        scale=EXPORT_SCALE_M,
        region=geom.bounds(),
        file_per_band=False
    )

    tif_to_large_png_rgba(tif_path, png_path, fig_inches=PNG_FIGSIZE, dpi=PNG_DPI)

    exported.append({
        "Plot": plot_id,
        "Landuse": landuse,
        "Sensor": "Sentinel-2",
        "Date": date_str,
        "Valid_area_m2": valid_area,
        "Valid_fraction": valid_frac,
        "TIFF": tif_path,
        "PNG": png_path
    })


# save log
log_df = pd.DataFrame(exported)
log_path = os.path.join(OUT_DIR, "S2_snapshot_export_log.csv")
log_df.to_csv(log_path, index=False)



# save legend
cmap = LinearSegmentedColormap.from_list("ndvi_yellow_green", PALETTE, N=256)
norm = Normalize(vmin=NDVI_MIN, vmax=NDVI_MAX)

fig = plt.figure(figsize=(1.8, 5.0))
ax = fig.add_axes([0.35, 0.08, 0.30, 0.84])

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = plt.colorbar(sm, cax=ax)
cbar.set_label("NDVI", rotation=90)
cbar.set_ticks([0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95])

legend_png = os.path.join(OUT_DIR, "S2_NDVI_legend_0p65_to_0p95.png")
plt.savefig(legend_png, dpi=400, bbox_inches="tight", transparent=True)
plt.close(fig)


# Phenology curves

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# area-based phenology curves
# sentinel-2 and landsat 8/9 on same plots
# figure 30: oil palm (bo5, ho2)
# figure 31: forest (bf2, bf3)


# plot settings
# use correct plot ids from dataset
PALM_PLOTS   = ["BO5", "HO2"]
FOREST_PLOTS = ["BF2", "BF3"]

NDVI_FLOOR_S2 = None
NDVI_FLOOR_L89 = 0.70

S2_COLOR = "#1E88E5"   # blue
L89_COLOR = "#D62728"  # red


# get area-based time series
def get_area_series(plot_id, sensor="S2", ndvi_floor=None):
    point_geom, circle_geom = get_plot_geometries(plot_id)

    if sensor == "S2":
        ic = s2_ndvi_strict.filterBounds(circle_geom).filterDate(start, end).sort("system:time_start")
        scale = 10
        fine_scale = 1
    else:
        ic = l89_ndvi_strict.filterBounds(circle_geom).filterDate(start, end).sort("system:time_start")
        scale = 30
        fine_scale = 1

    ts_area = area_weighted_ts(ic, circle_geom, scale=scale, fine_scale=fine_scale)

    # apply ndvi threshold if used
    if ndvi_floor is not None:
        ts_area = ts_area[ts_area["NDVI"] >= ndvi_floor].copy()

    wk_area = weekly_smooth(ts_area)
    return ts_area, wk_area


# single panel plot
def plot_phenology_panel(ax, plot_id, panel_label):
    landuse = plot_lookup.loc[plot_id, "Landuse"]

    ts_s2, wk_s2 = get_area_series(plot_id, sensor="S2", ndvi_floor=NDVI_FLOOR_S2)
    ts_l89, wk_l89 = get_area_series(plot_id, sensor="L89", ndvi_floor=NDVI_FLOOR_L89)


    # raw points (not in legend)
    if len(ts_s2) > 0:
        ax.plot(
            ts_s2["date"], ts_s2["NDVI"],
            ".", alpha=0.25, color=S2_COLOR,
            label="_nolegend_"
        )
    if len(ts_l89) > 0:
        ax.plot(
            ts_l89["date"], ts_l89["NDVI"],
            ".", alpha=0.25, color=L89_COLOR,
            label="_nolegend_"
        )

    # smoothed lines (shown in legend)
    if len(wk_s2) > 0:
        ax.plot(
            wk_s2["date"], wk_s2["NDVI_smooth"],
            "-", linewidth=1, color=S2_COLOR,
            label="Sentinel-2"
        )
    if len(wk_l89) > 0:
        ax.plot(
            wk_l89["date"], wk_l89["NDVI_smooth"],
            "-", linewidth=1, color=L89_COLOR,
            label="Landsat 8/9"
        )

    # axis formatting
    ax.set_xlabel("Month (2021)")
    ax.set_ylabel("NDVI")
    ax.set_ylim(0.65, 0.95)
    ax.set_xlim(pd.Timestamp("2021-02-01"), pd.Timestamp("2021-12-31"))

    # monthly ticks
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

    ax.set_title("")
    ax.grid(False)

    # legend with panel label
    leg = ax.legend(
        title=f"{panel_label} {plot_id} ({landuse})",
        frameon=True,
        loc="lower left",
        fontsize=8,
        title_fontsize=9
    )
    leg.get_frame().set_edgecolor("black")
    leg.get_frame().set_facecolor("white")
    leg.get_frame().set_alpha(1)


# two-panel figure
def make_two_panel_figure(plot_ids, save_name):
    fig, axes = plt.subplots(2, 1, figsize=(6, 6), sharex=True)

    panel_labels = ["(a)", "(b)"]

    for i, (ax, pid, panel_label) in enumerate(zip(axes, plot_ids, panel_labels)):
        if pid not in plot_lookup.index:
            ax.text(0.5, 0.5, f"{panel_label} {pid}\nPlot not found", ha="center", va="center")
            ax.set_axis_off()
        else:
            plot_phenology_panel(ax, pid, panel_label)

        # remove x labels on top panel
        if i == 0:
            ax.set_xlabel("")
            ax.set_xticklabels([])

    # keep x label on bottom panel
    axes[-1].set_xlabel("Month (2021)")

    plt.tight_layout()

    save_path = os.path.join(OUT, save_name)
    plt.savefig(save_path, dpi=400, bbox_inches="tight")

    plt.show()


# run figures

make_two_panel_figure(
    PALM_PLOTS,
    "Figure_30_oil_palm_phenology_area_based.png"
)


make_two_panel_figure(
    FOREST_PLOTS,
    "Figure_31_forest_phenology_area_based.png"
)


# red edge figures and tables

In [ ]:
# sentinel-2 red-edge indices using area-based extraction
import os
import numpy as np
import pandas as pd
import ee
from scipy.stats import linregress
import matplotlib.pyplot as plt

try:
    ee.Initialize(project="double-skyline-471010-g2")
except:
    ee.Authenticate()
    ee.Initialize(project="double-skyline-471010-g2")

# paths
BASE_FOLDER = "/content/drive/MyDrive/FINAL_ONE"
GROUND_PATH = os.path.join(BASE_FOLDER, "LAI_GROUND.csv")
OUT_PATH = os.path.join(BASE_FOLDER, "S2_STRICT_weighted_rededge.csv")
OUT_TABLE = os.path.join(BASE_FOLDER, "Table_rededge_model_performance_area_only.csv")
OUT_DIR = os.path.join(BASE_FOLDER, "FIG_rededge_scatter_area_only")
os.makedirs(OUT_DIR, exist_ok=True)

# load ground data
df = pd.read_csv(GROUND_PATH)

df["Plot"] = df["Plot"].astype(str).str.strip()
df["Landuse"] = df["Landuse"].astype(str).str.strip()
df["LAI"] = pd.to_numeric(df["LAI"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")

# use date_center if present
if "date_center" in df.columns:
    df["date_center"] = pd.to_datetime(df["date_center"], errors="coerce")
else:
    df["date_center"] = pd.to_datetime(df["Date"], errors="coerce")

df = df[df["Landuse"].isin(["Forest", "Oil palm"])].copy()
df = df.dropna(subset=["Plot", "Landuse", "LAI", "lon", "lat", "date_center"]).copy()

print("Rows kept:", len(df))
print(df["Landuse"].value_counts())

# build 1000 m2 plot geometry
PLOT_AREA_M2 = 1000.0
PLOT_RADIUS_M = float(np.sqrt(PLOT_AREA_M2 / np.pi))

def row_to_feature(r):
    point = ee.Geometry.Point([float(r["lon"]), float(r["lat"])])
    circle = point.buffer(PLOT_RADIUS_M)
    return ee.Feature(circle, {
        "Plot": str(r["Plot"]),
        "Landuse": str(r["Landuse"]),
        "LAI": float(r["LAI"]),
        "date_center": pd.to_datetime(r["date_center"]).strftime("%Y-%m-%d")
    })

fc = ee.FeatureCollection([row_to_feature(r) for _, r in df.iterrows()])
print("EE features:", fc.size().getInfo())

# sentinel-2 strict masking
S2_SR = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
S2_CPROB = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")

MAX_SCENE_CLOUD_PCT = 80
CLOUD_PROB_THR = 40
DILATE_METERS = 60
MIN_VALID = 2

join = ee.Join.saveFirst("cloud_prob_img")
join_filter = ee.Filter.equals(leftField="system:index", rightField="system:index")
s2_joined = join.apply(S2_SR, S2_CPROB, join_filter)

def strict_mask_s2(img):
    scl = img.select("SCL")
    good = (
        scl.neq(0)
        .And(scl.neq(1))
        .And(scl.neq(2))
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    qa = img.select("QA60")
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11
    qa_clear = qa.bitwiseAnd(cloud_bit).eq(0).And(
        qa.bitwiseAnd(cirrus_bit).eq(0)
    )

    cprob = ee.Image(img.get("cloud_prob_img")).select("probability")
    cprob_clear = cprob.lt(CLOUD_PROB_THR)

    mask = good.And(qa_clear).And(cprob_clear)

    if DILATE_METERS > 0:
        bad = mask.Not()
        bad = bad.focal_max(radius=DILATE_METERS, units="meters")
        mask = bad.Not()

    return img.updateMask(mask)

def add_indices(img):
    red = img.select("B4").multiply(0.0001)
    nir = img.select("B8").multiply(0.0001)
    re1 = img.select("B5").multiply(0.0001)
    re2 = img.select("B6").multiply(0.0001)
    re3 = img.select("B7").multiply(0.0001)

    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI")
    ndre = nir.subtract(re1).divide(nir.add(re1)).rename("NDRE705")
    cire = nir.divide(re1).subtract(1).rename("CIre705")
    mtci = re3.subtract(re1).divide(re2.subtract(re1)).rename("MTCI")

    return img.addBands([ndvi, ndre, cire, mtci]).copyProperties(
        img, ["system:time_start", "CLOUDY_PIXEL_PERCENTAGE"]
    )

s2_strict_idx = (
    ee.ImageCollection(s2_joined)
    .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", MAX_SCENE_CLOUD_PCT))
    .map(strict_mask_s2)
    .map(add_indices)
    .select(["NDVI", "NDRE705", "CIre705", "MTCI"])
)

# area-weighted extraction helpers
def weighted_value_for_band(img, geom, band_name, scale=10, fine_scale=1):
    band = img.select(band_name)
    proj = band.projection()

    inside_fine = (
        ee.Image.constant(1)
        .toFloat()
        .clip(geom)
        .reproject(crs=proj, scale=fine_scale)
    )

    inside_frac = (
        inside_fine
        .reduceResolution(
            reducer=ee.Reducer.mean(),
            maxPixels=4096
        )
        .reproject(crs=proj, scale=scale)
        .rename("inside_frac")
    )

    pixel_area = ee.Image.pixelArea().reproject(crs=proj, scale=scale)
    weights = inside_frac.multiply(pixel_area).rename("weights")
    valid_weights = weights.updateMask(band.mask()).rename("weights")
    weighted_band = band.multiply(valid_weights).rename("weighted_band")

    stats = ee.Image.cat([weighted_band, valid_weights]).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geom,
        scale=scale,
        crs=proj,
        maxPixels=1e7
    )

    num_raw = stats.get("weighted_band")
    den_raw = stats.get("weights")

    num = ee.Number(ee.Algorithms.If(num_raw, num_raw, 0))
    den = ee.Number(ee.Algorithms.If(den_raw, den_raw, 0))

    return ee.Algorithms.If(den.gt(0), num.divide(den), None)

def per_image_weighted_index_fc(ic, geom, band_name):
    def sample_one(img):
        v = weighted_value_for_band(img, geom, band_name, scale=10, fine_scale=1)
        return ee.Feature(None, {"v": v})
    return ic.map(sample_one).filter(ee.Filter.notNull(["v"]))

def median_over_valid_dates(ic, geom, band_name):
    per_img = per_image_weighted_index_fc(ic, geom, band_name)
    n_valid = per_img.size()
    v_list = ee.List(per_img.aggregate_array("v"))

    v_med = ee.Algorithms.If(
        n_valid.gt(0),
        ee.Number(v_list.reduce(ee.Reducer.median())),
        None
    )

    return n_valid, v_med

# fallback windows based on valid ndvi dates
def add_idx_vals(feat):
    center = ee.Date(feat.get("date_center"))
    geom = feat.geometry()

    def window_collection(days):
        days = ee.Number(days)
        return (
            s2_strict_idx
            .filterBounds(geom)
            .filterDate(
                center.advance(days.multiply(-1), "day"),
                center.advance(days, "day")
            )
        )

    ic15 = window_collection(15)
    ic30 = window_collection(30)
    ic50 = window_collection(50)

    n15 = ic15.size()
    n30 = ic30.size()
    n50 = ic50.size()

    nvalid15, ndvi15 = median_over_valid_dates(ic15, geom, "NDVI")
    nvalid30, ndvi30 = median_over_valid_dates(ic30, geom, "NDVI")
    nvalid50, ndvi50 = median_over_valid_dates(ic50, geom, "NDVI")

    _, ndre15 = median_over_valid_dates(ic15, geom, "NDRE705")
    _, cire15 = median_over_valid_dates(ic15, geom, "CIre705")
    _, mtci15 = median_over_valid_dates(ic15, geom, "MTCI")

    _, ndre30 = median_over_valid_dates(ic30, geom, "NDRE705")
    _, cire30 = median_over_valid_dates(ic30, geom, "CIre705")
    _, mtci30 = median_over_valid_dates(ic30, geom, "MTCI")

    _, ndre50 = median_over_valid_dates(ic50, geom, "NDRE705")
    _, cire50 = median_over_valid_dates(ic50, geom, "CIre705")
    _, mtci50 = median_over_valid_dates(ic50, geom, "MTCI")

    return ee.Algorithms.If(
        ee.Number(nvalid15).gte(MIN_VALID),
        feat.set({
            "window_days_used": 15,
            "n_images_in_window": n15,
            "n_valid_ndvi": nvalid15,
            "NDVI_weighted": ndvi15,
            "NDRE705_weighted": ndre15,
            "CIre705_weighted": cire15,
            "MTCI_weighted": mtci15
        }),
        ee.Algorithms.If(
            ee.Number(nvalid30).gte(MIN_VALID),
            feat.set({
                "window_days_used": 30,
                "n_images_in_window": n30,
                "n_valid_ndvi": nvalid30,
                "NDVI_weighted": ndvi30,
                "NDRE705_weighted": ndre30,
                "CIre705_weighted": cire30,
                "MTCI_weighted": mtci30
            }),
            ee.Algorithms.If(
                ee.Number(nvalid50).gte(MIN_VALID),
                feat.set({
                    "window_days_used": 50,
                    "n_images_in_window": n50,
                    "n_valid_ndvi": nvalid50,
                    "NDVI_weighted": ndvi50,
                    "NDRE705_weighted": ndre50,
                    "CIre705_weighted": cire50,
                    "MTCI_weighted": mtci50
                }),
                feat.set({
                    "window_days_used": None,
                    "n_images_in_window": 0,
                    "n_valid_ndvi": 0,
                    "NDVI_weighted": None,
                    "NDRE705_weighted": None,
                    "CIre705_weighted": None,
                    "MTCI_weighted": None
                })
            )
        )
    )

out_fc = fc.map(lambda f: ee.Feature(add_idx_vals(f)))

print("Output feature count:", out_fc.size().getInfo())
print(out_fc.first().toDictionary().getInfo())

# convert to pandas
data = pd.DataFrame([f["properties"] for f in out_fc.getInfo()["features"]])

for col in [
    "LAI", "window_days_used", "n_images_in_window", "n_valid_ndvi",
    "NDVI_weighted", "NDRE705_weighted", "CIre705_weighted", "MTCI_weighted"
]:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

data["Landuse"] = data["Landuse"].astype(str).str.strip()
data.to_csv(OUT_PATH, index=False)

print("Saved to:", OUT_PATH)
print("Rows with valid NDRE705:", data.dropna(subset=["LAI", "NDRE705_weighted"]).shape[0])
print("Rows with valid CIre705:", data.dropna(subset=["LAI", "CIre705_weighted"]).shape[0])
print("Rows with valid MTCI:", data.dropna(subset=["LAI", "MTCI_weighted"]).shape[0])

# model performance table for red-edge index-lai relationships
# area-based only

area = pd.read_csv(OUT_PATH)

area["Plot"] = area["Plot"].astype(str).str.strip()
area["Landuse"] = area["Landuse"].astype(str).str.strip()
area["LAI"] = pd.to_numeric(area["LAI"], errors="coerce")

for col in ["NDRE705_weighted", "CIre705_weighted", "MTCI_weighted"]:
    if col in area.columns:
        area[col] = pd.to_numeric(area[col], errors="coerce")

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

rows = []

index_specs = [
    ("NDRE705", "NDRE705_weighted"),
    ("CIre705", "CIre705_weighted"),
    ("MTCI", "MTCI_weighted"),
]

for index_name, area_col in index_specs:
    for landuse in ["Forest", "Oil palm"]:
        sub = area[(area["Landuse"] == landuse)].dropna(subset=["LAI", area_col]).copy()

        if len(sub) >= 2:
            x = sub[area_col].values
            y = sub["LAI"].values
            slope, intercept, r, p, se = linregress(x, y)
            yhat = slope * x + intercept
            r2 = r**2
            this_rmse = rmse(y, yhat)
        else:
            r2 = np.nan
            this_rmse = np.nan

        rows.append({
            "Index": index_name,
            "Land use": landuse,
            "R²": r2,
            "RMSE": this_rmse,
            "n": len(sub)
        })

table = pd.DataFrame(rows)
table["R²"] = table["R²"].round(3)
table["RMSE"] = table["RMSE"].round(3)

print(table.to_string(index=False))
table.to_csv(OUT_TABLE, index=False)
print("\nSaved:", OUT_TABLE)

# scatter plots for red-edge index vs lai
# area-based only
# one figure per index
# panels are land use

forest_color = "#2E7D32"
oilpalm_color = "#F28E2B"

index_specs = [
    ("NDRE705", "NDRE705_weighted"),
    ("CIre705", "CIre705_weighted"),
    ("MTCI", "MTCI_weighted"),
]

label_map = {
    "NDRE705": r"NDRE$_{705}$",
    "CIre705": r"CIre$_{705}$",
    "MTCI": "MTCI"
}

for index_name, area_col in index_specs:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True)

    for ax, landuse, color in zip(
        axes,
        ["Forest", "Oil palm"],
        [forest_color, oilpalm_color]
    ):
        sub = area[area["Landuse"] == landuse].dropna(subset=["LAI", area_col]).copy()

        if len(sub) > 0:
            x = sub[area_col].values
            y = sub["LAI"].values
            ax.scatter(
                x, y,
                color=color,
                edgecolor="black",
                linewidth=0.4,
                s=28,
                alpha=0.85
            )

            if len(sub) >= 2:
                slope, intercept, r, p, se = linregress(x, y)
                x_line = np.linspace(np.nanmin(x), np.nanmax(x), 100)
                y_line = slope * x_line + intercept
                ax.plot(x_line, y_line, color=color, linewidth=1.2)
                txt = f"R² = {r**2:.2f}\nn = {len(sub)}"
            else:
                txt = f"n = {len(sub)}"
        else:
            txt = "n = 0"

        ax.text(
            0.04, 0.96,
            txt,
            transform=ax.transAxes,
            va="top",
            fontsize=8,
            bbox=dict(boxstyle="round", facecolor="white", edgecolor="black", alpha=0.9)
        )

        ax.set_xlabel(label_map[index_name])
        ax.set_title(landuse)
        ax.grid(False)

    axes[0].set_ylabel("Ground LAI")

    plt.tight_layout()

    out_png = os.path.join(OUT_DIR, f"{index_name}_area_scatter.png")
    out_pdf = os.path.join(OUT_DIR, f"{index_name}_area_scatter.pdf")

    plt.savefig(out_png, dpi=400, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()



# Masking pixel counts / area covered


In [ ]:
# masking and pixel support diagnostics
# (a) valid area fraction after masking
# (b) true intersecting pixel count
# all plots combined
# sentinel-2 = green
# landsat 8/9 = blue

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

BASE = "/content/drive/MyDrive/FINAL_ONE/"

# load files
s2 = pd.read_csv(os.path.join(BASE, "S2_masking_quality_true_pixelcounts.csv"))
l89 = pd.read_csv(os.path.join(BASE, "L89_NDVI_weighted_1000m2_truepixelcounts.csv"))
counts = pd.read_csv(os.path.join(BASE, "pixel_counts_true_geometric.csv"))

for df in [s2, l89, counts]:
    df["Plot"] = df["Plot"].astype(str).str.strip()
    if "Landuse" in df.columns:
        df["Landuse"] = df["Landuse"].astype(str).str.strip()

# prepare variables
s2["valid_area_fraction"] = pd.to_numeric(s2["valid_area_m2_median_date"], errors="coerce") / 1000.0
l89["valid_area_fraction"] = pd.to_numeric(l89["valid_area_m2_median_date"], errors="coerce") / 1000.0

s2_area_vals = s2["valid_area_fraction"].dropna().values
l89_area_vals = l89["valid_area_fraction"].dropna().values

s2_pix_vals = pd.to_numeric(counts["S2_true_pixel_count"], errors="coerce").dropna().values
l89_pix_vals = pd.to_numeric(counts["L89_true_pixel_count"], errors="coerce").dropna().values

sentinel_color = "#66BB6A"   # light-medium green
landsat_color = "#1E88E5"    # blue

legend_handles = [
    Patch(facecolor=sentinel_color, edgecolor="black", label="Sentinel-2"),
    Patch(facecolor=landsat_color, edgecolor="black", label="Landsat 8/9")
]

# histogram helper
def plot_overlap_hist(ax, vals1, vals2, bins, color1, color2):
    c1, edges = np.histogram(vals1, bins=bins)
    c2, _ = np.histogram(vals2, bins=bins)

    p1 = c1 / c1.sum() if c1.sum() > 0 else np.zeros_like(c1, dtype=float)
    p2 = c2 / c2.sum() if c2.sum() > 0 else np.zeros_like(c2, dtype=float)

    lefts = edges[:-1]
    width = np.diff(edges)

    for i in range(len(lefts)):
        if p1[i] >= p2[i]:
            draw_order = [(p1[i], color1), (p2[i], color2)]
        else:
            draw_order = [(p2[i], color2), (p1[i], color1)]

        for h, color in draw_order:
            if h > 0:
                ax.bar(
                    lefts[i],
                    h,
                    width=width[i],
                    align="edge",
                    color=color,
                    edgecolor="black",
                    linewidth=0.6,
                    alpha=1.0
                )

# make figure
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

# panel a: valid area fraction
bins_area = np.linspace(0.4, 1.0, 16)
plot_overlap_hist(
    axes[0],
    s2_area_vals,
    l89_area_vals,
    bins=bins_area,
    color1=sentinel_color,
    color2=landsat_color
)
axes[0].set_xlim(0.4, 1.0)
axes[0].set_xlabel("Valid area fraction")
axes[0].set_ylabel("Proportion of plots")
axes[0].set_title("(a) Valid area fraction")
axes[0].grid(False)

leg0 = axes[0].legend(handles=legend_handles, frameon=True, fontsize=8, loc="upper left")
leg0.get_frame().set_edgecolor("black")
leg0.get_frame().set_facecolor("white")
leg0.get_frame().set_alpha(1)

# panel b: true intersecting pixel count
overall_max = int(max(np.nanmax(s2_pix_vals), np.nanmax(l89_pix_vals)))
bins_pix = np.arange(-0.5, overall_max + 1.5, 1)

plot_overlap_hist(
    axes[1],
    s2_pix_vals,
    l89_pix_vals,
    bins=bins_pix,
    color1=sentinel_color,
    color2=landsat_color
)
axes[1].set_xlabel("Intersecting pixel count")
axes[1].set_ylabel("Proportion of plots")
axes[1].set_title("(b) Intersecting pixel count")
axes[1].grid(False)

leg1 = axes[1].legend(handles=legend_handles, frameon=True, fontsize=8, loc="upper right")
leg1.get_frame().set_edgecolor("black")
leg1.get_frame().set_facecolor("white")
leg1.get_frame().set_alpha(1)

for ax in axes:
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")

plt.tight_layout()
out_path = os.path.join(BASE, "FIG_4_2_masking_pixel_support_combined.png")
plt.savefig(out_path, dpi=400, bbox_inches="tight")
plt.show()



# temporal stretching

In [ ]:
# temporal window usage figures


import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

BASE = "/content/drive/MyDrive/FINAL_ONE/"

# load files
s2 = pd.read_csv(os.path.join(BASE,"S2_NDVI_areawtd_STRICTmask.csv"))
l89 = pd.read_csv(os.path.join(BASE,"L89_NDVI_weighted_1000m2_STRICTmask.csv"))

# clean data
for df in [s2, l89]:
    df["Landuse"] = df["Landuse"].astype(str).str.strip()
    df["window_days_used"] = pd.to_numeric(df["window_days_used"], errors="coerce")

s2 = s2.dropna(subset=["NDVI_S2"])
l89 = l89.dropna(subset=["NDVI_L89"])

# calculate percentages
def compute_props(df):
    counts = (
        df.groupby(["Landuse","window_days_used"])
        .size()
        .reset_index(name="count")
    )

    totals = df.groupby("Landuse").size().reset_index(name="total")

    summary = counts.merge(totals, on="Landuse")
    summary["percent"] = 100 * summary["count"] / summary["total"]

    return summary

s2_summary = compute_props(s2)
l89_summary = compute_props(l89)

# colours
forest_color = "#2E7D32"
palm_color   = "#F57C00"

# y-axis settings
ymin, ymax = 0, 60
yticks = np.arange(0, 61, 10)

# figure 1: sentinel-2
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

for i, lc in enumerate(["Forest", "Oil palm"]):

    sub = s2_summary[s2_summary["Landuse"] == lc]
    color = forest_color if lc == "Forest" else palm_color

    axes[i].bar(
        sub["window_days_used"].astype(int).astype(str),
        sub["percent"],
        color=color,
        edgecolor="black",
        linewidth=1.0
    )

    # y-axis label
    if lc == "Forest":
        axes[i].set_ylabel("Forest percent of plots")
    else:
        axes[i].set_ylabel("Oil palm percent of plots")

    axes[i].set_xlabel("Sentinel-2 temporal window (days)")

    # fixed y-axis
    axes[i].set_ylim(ymin, ymax)
    axes[i].set_yticks(yticks)

    axes[i].grid(False)

    for spine in axes[i].spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.0)
        spine.set_color("black")

plt.tight_layout()
plt.show()

# figure 2: landsat
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

for i, lc in enumerate(["Forest", "Oil palm"]):

    sub = l89_summary[l89_summary["Landuse"] == lc]
    color = forest_color if lc == "Forest" else palm_color

    axes[i].bar(
        sub["window_days_used"].astype(int).astype(str),
        sub["percent"],
        color=color,
        edgecolor="black",
        linewidth=1.0
    )

    # y-axis label
    if lc == "Forest":
        axes[i].set_ylabel("Forest percent of plots")
    else:
        axes[i].set_ylabel("Oil palm percent of plots")

    axes[i].set_xlabel("Landsat 8/9 temporal window (days)")

    # fixed y-axis
    axes[i].set_ylim(ymin, ymax)
    axes[i].set_yticks(yticks)

    axes[i].grid(False)

    for spine in axes[i].spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.0)
        spine.set_color("black")

plt.tight_layout()
plt.show()

# cloud probability histogram

In [ ]:
# extract cloud probability values for all plots

records = []

for i, row in df.iterrows():
    records.extend(get_plot_cloud_probs(row, window_days=WINDOW_DAYS_FOR_FIG))

    if (i + 1) % 10 == 0 or (i + 1) == len(df):
        print(f"processed {i+1}/{len(df)} plots; rows: {len(records)}")

cloud_df = pd.DataFrame(records)
cloud_df.to_csv(OUT_CSV, index=False)

print("saved:", OUT_CSV)
print("rows:", len(cloud_df))


# summary stats
summary = cloud_df["cloud_prob"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
print("\nsummary:")
print(summary)


# threshold split
below = (cloud_df["cloud_prob"] < CLOUD_PROB_THR).sum()
above = (cloud_df["cloud_prob"] >= CLOUD_PROB_THR).sum()
total = len(cloud_df)

print(f"\nbelow threshold (<{CLOUD_PROB_THR}%): {below} ({100*below/total:.1f}%)")
print(f"above threshold (≥{CLOUD_PROB_THR}%): {above} ({100*above/total:.1f}%)")
# clean cloud probability histogram

plt.figure(figsize=(7, 4.8))

plt.hist(cloud_df["cloud_prob"], bins=np.arange(0, 101, 5), edgecolor="black")

# threshold line
plt.axvline(CLOUD_PROB_THR, linestyle="--", linewidth=1.5)

plt.xlabel("s2cloudless cloud probability (%)")
plt.ylabel("Pixel count")
plt.title("Distribution of cloud probability values across candidate Sentinel-2 plot pixels")

plt.tight_layout()
plt.show()

# ground data figures

In [ ]:
# Density plot of ground-measured LAI by land use
# Forest = green, Oil palm = orange
# Includes shaded overlap range with dashed boundaries
# Full rectangular outline box, no grid

use_kde = True
try:
    from scipy.stats import gaussian_kde
except Exception:
    use_kde = False
    print("scipy not available for KDE; using histogram density fallback.")

# Colours
forest_color = "#2E7D32"   # green
palm_color   = "#F57C00"   # orange

# Shared x-range across both land uses
xmin = min(forest.min(), palm.min())
xmax = max(forest.max(), palm.max())
xgrid = np.linspace(xmin, xmax, 400)

plt.figure(figsize=(6.2, 4.2))
ax = plt.gca()

if use_kde:
    kde_f = gaussian_kde(forest)
    kde_p = gaussian_kde(palm)

    y_f = kde_f(xgrid)
    y_p = kde_p(xgrid)

    ax.plot(xgrid, y_f, color=forest_color, linewidth=2.0, label="Forest")
    ax.plot(xgrid, y_p, color=palm_color, linewidth=2.0, label="Oil palm")

    ax.fill_between(xgrid, y_f, color=forest_color, alpha=0.15)
    ax.fill_between(xgrid, y_p, color=palm_color, alpha=0.15)

else:
    # Histogram density fallback
    bins = 25
    y_f, edges = np.histogram(forest, bins=bins, range=(xmin, xmax), density=True)
    y_p, _ = np.histogram(palm, bins=bins, range=(xmin, xmax), density=True)
    centers = 0.5 * (edges[1:] + edges[:-1])

    ax.plot(centers, y_f, color=forest_color, linewidth=2.0, label="Forest")
    ax.plot(centers, y_p, color=palm_color, linewidth=2.0, label="Oil palm")

    ax.fill_between(centers, y_f, color=forest_color, alpha=0.15)
    ax.fill_between(centers, y_p, color=palm_color, alpha=0.15)

# Highlight overlap range
if has_overlap:
    ax.axvspan(overlap_low, overlap_high, color="grey", alpha=0.22, zorder=0)
    ax.axvline(overlap_low, color="black", linestyle="--", linewidth=1.0)
    ax.axvline(overlap_high, color="black", linestyle="--", linewidth=1.0)

    ymax = ax.get_ylim()[1]
    x_mid = (overlap_low + overlap_high) / 2
    x_shift = 0.04 * (xmax - xmin)

    ax.text(
        x_mid + x_shift,
        ymax * 0.92,
        "Overlap range",
        ha="left",
        va="top",
        fontsize=9
    )

# Labels and axis formatting
ax.set_xlabel("LAI (m$^2$ m$^{-2}$)")
ax.set_ylabel("Density")
ax.set_title("Density of ground-measured LAI by land use")

ax.set_xlim(xmin - 0.1, xmax + 0.1)
ax.set_ylim(bottom=0)
ax.margins(x=0, y=0)

# Full rectangular outline box
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.0)
    spine.set_color("black")

# No grid
ax.legend(frameon=False)

plt.tight_layout()

png_path = os.path.join(OUT_DIR, "Fig_4_3_Ground_LAI_density_KDE.png")
pdf_path = os.path.join(OUT_DIR, "Fig_4_3_Ground_LAI_density_KDE.pdf")

plt.savefig(png_path, dpi=400, bbox_inches="tight")
plt.savefig(pdf_path, bbox_inches="tight")
plt.show()



In [ ]:
# lai density plot by land use
# forest = green
# oil palm = orange
# overlap range is shaded

use_kde = True
try:
    from scipy.stats import gaussian_kde
except Exception:
    use_kde = False
    print("scipy not available for KDE; using histogram density fallback.")

# colours
forest_color = "#2E7D32"
palm_color   = "#F57C00"

# shared x-range
xmin = min(forest.min(), palm.min())
xmax = max(forest.max(), palm.max())
xgrid = np.linspace(xmin, xmax, 400)

plt.figure(figsize=(6.2, 4.2))
ax = plt.gca()

if use_kde:
    kde_f = gaussian_kde(forest)
    kde_p = gaussian_kde(palm)

    y_f = kde_f(xgrid)
    y_p = kde_p(xgrid)

    ax.plot(xgrid, y_f, color=forest_color, linewidth=2.0, label="Forest")
    ax.plot(xgrid, y_p, color=palm_color, linewidth=2.0, label="Oil palm")

    ax.fill_between(xgrid, y_f, color=forest_color, alpha=0.15)
    ax.fill_between(xgrid, y_p, color=palm_color, alpha=0.15)

else:
    # histogram fallback
    bins = 25
    y_f, edges = np.histogram(forest, bins=bins, range=(xmin, xmax), density=True)
    y_p, _ = np.histogram(palm, bins=bins, range=(xmin, xmax), density=True)
    centers = 0.5 * (edges[1:] + edges[:-1])

    ax.plot(centers, y_f, color=forest_color, linewidth=2.0, label="Forest")
    ax.plot(centers, y_p, color=palm_color, linewidth=2.0, label="Oil palm")

    ax.fill_between(centers, y_f, color=forest_color, alpha=0.15)
    ax.fill_between(centers, y_p, color=palm_color, alpha=0.15)

# overlap range
if has_overlap:
    ax.axvspan(overlap_low, overlap_high, color="grey", alpha=0.22, zorder=0)
    ax.axvline(overlap_low, color="black", linestyle="--", linewidth=1.0)
    ax.axvline(overlap_high, color="black", linestyle="--", linewidth=1.0)

    ymax = ax.get_ylim()[1]
    x_mid = (overlap_low + overlap_high) / 2
    x_shift = 0.04 * (xmax - xmin)

    ax.text(
        x_mid + x_shift,
        ymax * 0.92,
        "Overlap range",
        ha="left",
        va="top",
        fontsize=9
    )

# labels
ax.set_xlabel("LAI (m$^2$ m$^{-2}$)")
ax.set_ylabel("Density")
ax.set_title("Density of ground-measured LAI by land use")

ax.set_xlim(xmin - 0.1, xmax + 0.1)
ax.set_ylim(bottom=0)
ax.margins(x=0, y=0)

# show full box
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.0)
    spine.set_color("black")

ax.legend(frameon=False)

plt.tight_layout()

png_path = os.path.join(OUT_DIR, "Fig_4_3_Ground_LAI_density_KDE.png")
pdf_path = os.path.join(OUT_DIR, "Fig_4_3_Ground_LAI_density_KDE.pdf")

plt.savefig(png_path, dpi=400, bbox_inches="tight")
plt.savefig(pdf_path, bbox_inches="tight")
plt.show()

